# Sparse Rebasin — ResNet20 ×4 / CIFAR-100 på Colab


## 1. Afhængigheder og GPU


In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet",
     "PyYAML>=6", "scipy>=1.13", "tqdm>=4.41", "matplotlib>=3.7"],
    check=True,
)

import torch
import torchvision
import numpy as np
import scipy
import yaml
from torch.utils.flop_counter import FlopCounterMode

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__, "| torchvision:", torchvision.__version__)
print("CUDA build:", torch.version.cuda)
if not torch.cuda.is_available():
    raise RuntimeError("Ingen CUDA-GPU. Vælg GPU som hardwareaccelerator og kør igen.")
print("GPU:", torch.cuda.get_device_name(0))
print("GPU-hukommelse (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))


## 2. Genskab projektets filer


In [ ]:
import os
from pathlib import Path

PROJECT_DIR = Path("/content/sparse-rebasin-progressive-growth")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
(PROJECT_DIR / "configs").mkdir(exist_ok=True)
(PROJECT_DIR / "resnet50_src").mkdir(exist_ok=True)
os.chdir(PROJECT_DIR)
print("Arbejdsmappe:", PROJECT_DIR)


In [ ]:
%%writefile models.py
"""
Model definition taken from REPAIR. (Jordan et al. 2022)
"""

import torch.nn as nn
import torch.nn.functional as F
import torch.nn.init as init
import torchvision.models

dataset_num_classes = {
    "cifar10": 10,
    "cifar100": 100,
    "mnist": 10,
    "svhn": 10,
}

cfg = {
    "VGG11": {
        "layers": [64, "M", 128, "M", 256, 256, "M", 512, 512, "M", 512, 512, "M"]
    }
}


class VGG11_nofc(nn.Module):
    def __init__(self, config, vgg_name, num_classes, w=1, init_weights=True):
        super(VGG11_nofc, self).__init__()
        self.config = config
        self.w = w
        self.num_classes = num_classes
        print("the number of classes", self.num_classes)
        self.features = self._make_layers(cfg[vgg_name]["layers"])
        self.classifier = nn.Linear(self.w * 512, self.num_classes)
        if init_weights:
            self._initialize_weights()

    def forward(self, x):
        out = self.features(x)
        out = out.view(out.size(0), -1)
        out = self.classifier(out)
        return out

    def _make_layers(self, cfg):
        layers = []
        in_channels = 1 if self.config["dataset"] == "mnist" else 3
        for x in cfg:
            if x == "M":
                # MNIST's 28x28 input reaches odd spatial sizes before the
                # final VGG pooling layer. Ceil mode keeps those sizes valid.
                layers += [nn.MaxPool2d(kernel_size=2, stride=2, ceil_mode=True)]
            else:
                layers.append(
                    nn.Conv2d(
                        in_channels if in_channels == 3 else self.w * in_channels,
                        self.w * x,
                        kernel_size=3,
                        padding=1,
                    )
                )
                layers.append(nn.ReLU(inplace=True))
                in_channels = x
        layers += [nn.AvgPool2d(kernel_size=1, stride=1)]
        return nn.Sequential(*layers)

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                print("Initializing with Kaiming Normal", m)
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

            elif isinstance(m, nn.Linear):
                print("Initializing with Normal", m)
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)


class LeNet(nn.Module):
    def __init__(self, config, w=1, init_weights=True):
        super().__init__()
        num_classes = dataset_num_classes.get(config["dataset"])
        if num_classes is None:
            raise ValueError(f"Unsupported dataset: {config['dataset']}")

        input_channels = 1 if config["dataset"] == "mnist" else 3
        self.features = nn.Sequential(
            nn.Conv2d(input_channels, 6 * w, kernel_size=5),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(6 * w, 16 * w, kernel_size=5),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        image_size = 28 if config["dataset"] == "mnist" else 32
        feature_size = (image_size - 4) // 2
        feature_size = (feature_size - 4) // 2
        self.classifier = nn.Linear(16 * w * feature_size * feature_size, num_classes)

        if init_weights:
            self.apply(_weights_init)

    def forward(self, x):
        out = self.features(x)
        out = out.reshape(out.size(0), -1)
        return self.classifier(out)


###############################################################################################################################################


def _weights_init(m):
    classname = m.__class__.__name__
    if isinstance(m, nn.Linear) or isinstance(m, nn.Conv2d):
        init.kaiming_normal_(m.weight)


class LambdaLayer(nn.Module):
    def __init__(self, lambd):
        super(LambdaLayer, self).__init__()
        self.lambd = lambd

    def forward(self, x):
        return self.lambd(x)


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(
            in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False
        )
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(
            planes, planes, kernel_size=3, stride=1, padding=1, bias=False
        )
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            """
            For CIFAR10 ResNet paper uses:
            
            self.shortcut = LambdaLayer(lambda x:
                                        F.pad(x[:, :, ::2, ::2], (0, 0, 0, 0, planes//4, planes//4), "constant", 0))
            """
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_planes,
                    planes,
                    kernel_size=3,
                    stride=stride,
                    padding=1,
                    bias=False,
                ),
                nn.BatchNorm2d(planes),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out


class ResNet(nn.Module):
    def __init__(self, block, num_blocks, config, w=1):
        super(ResNet, self).__init__()

        self.num_classes = dataset_num_classes.get(config["dataset"])
        if self.num_classes is None:
            raise ValueError(f"Unsupported dataset: {config['dataset']}")
        input_channels = 1 if config["dataset"] == "mnist" else 3
        self.in_planes = w * 16
        self.conv1 = nn.Conv2d(
            input_channels, w * 16, kernel_size=3, stride=1, padding=1, bias=False
        )
        self.bn1 = nn.BatchNorm2d(w * 16)
        self.layer1 = self._make_layer(block, w * 16, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, w * 32, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, w * 64, num_blocks[2], stride=2)
        print("the number of classes", self.num_classes)
        self.linear = nn.Linear(w * 64, self.num_classes)

        self.apply(_weights_init)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_planes, planes, stride))
            self.in_planes = planes * block.expansion

        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = F.avg_pool2d(out, out.size()[3])
        out = out.view(out.size(0), -1)
        out = self.linear(out)
        return out


###############################################################################################################################################


def resnet50():
    model = torchvision.models.resnet50(weights=None)
    return model.cuda()


def get_blocks(net):
    return nn.Sequential(
        nn.Sequential(net.conv1, net.bn1, net.relu, net.maxpool),
        *net.layer1,
        *net.layer2,
        *net.layer3,
        *net.layer4,
    )


In [ ]:
%%writefile loaders.py
import torch
from torchvision import datasets, transforms
import os
import yaml

def cifar_dataloader(batch_size, config, slurm_tmpdir = None):

    dataset = config["dataset"]
    dataset_root = config["dataset_root"][dataset]

    if dataset == "cifar10":
        print("Load CIFAR10 dataset")
        normalize = transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
        train_transform = transforms.Compose(
            [
                transforms.RandomHorizontalFlip(),
                transforms.RandomCrop(32, 4),
                transforms.ToTensor(),
                normalize,
            ]
        )
        test_transform = transforms.Compose(
            [
                transforms.ToTensor(),
                normalize,
            ]
        )
        train_ds = datasets.CIFAR10(
            root=dataset_root,
            train=True,
            download=True,
            transform=train_transform,
        )
        test_ds = datasets.CIFAR10(
            root=dataset_root,
            train=False,
            download=True,
            transform=test_transform,
        )
    elif dataset == "cifar100":
        print("Load CIFAR100 dataset")
        normalize = transforms.Normalize((0.5071, 0.4865, 0.4409), (0.2673, 0.2564, 0.2762))
        train_transform = transforms.Compose(
            [
                transforms.RandomHorizontalFlip(),
                transforms.RandomCrop(32, 4),
                transforms.ToTensor(),
                normalize,
            ]
        )
        test_transform = transforms.Compose(
            [
                transforms.ToTensor(),
                normalize,
            ]
        )
        train_ds = datasets.CIFAR100(
            root=dataset_root,
            train=True,
            download=True,
            transform=train_transform,
        )
        test_ds = datasets.CIFAR100(
            root=dataset_root,
            train=False,
            download=True,
            transform=test_transform,
        )

    elif dataset == "imagenet":
        print("Load ImageNet dataset")
        traindir = os.path.join(slurm_tmpdir, 'train')
        valdir = os.path.join(slurm_tmpdir, 'val')
        normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                     std=[0.229, 0.224, 0.225])

        train_ds = datasets.ImageFolder(
            traindir,
            transforms.Compose([
                transforms.RandomResizedCrop(224),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                normalize,
            ]))

        test_ds = datasets.ImageFolder(
            valdir,
            transforms.Compose([
                transforms.Resize(256),
                transforms.CenterCrop(224),
                transforms.ToTensor(),
                normalize,
            ]))

    elif dataset == "svhn":
        print("Load SVHN dataset")
        normalize = transforms.Normalize((0.4377, 0.4438, 0.4728), (0.1980, 0.2010, 0.1970))
        train_transform = transforms.Compose(
            [
                transforms.RandomCrop(32, padding=4),
                transforms.ToTensor(),
                normalize,
            ]
        )
        test_transform = transforms.Compose(
            [
                transforms.ToTensor(),
                normalize,
            ]
        )
        train_ds = datasets.SVHN(
            root=dataset_root,
            split="train",
            download=False,
            transform=train_transform,
        )
        test_ds = datasets.SVHN(
            root=dataset_root,
            split="test",
            download=False,
            transform=test_transform,
        )
    elif dataset == "mnist":
        print("Load MNIST dataset")
        normalize = transforms.Normalize((0.1307,), (0.3081,))
        train_transform = transforms.Compose(
            [
                transforms.RandomCrop(28, padding=2),
                transforms.ToTensor(),
                normalize,
            ]
        )
        test_transform = transforms.Compose(
            [
                transforms.ToTensor(),
                normalize,
            ]
        )
        train_ds = datasets.MNIST(
            root=dataset_root,
            train=True,
            download=True,
            transform=train_transform,
        )
        test_ds = datasets.MNIST(
            root=dataset_root,
            train=False,
            download=True,
            transform=test_transform,
        )
    else:
        raise ValueError(f"Unsupported dataset: {dataset}")

    num_workers = config.get("num_workers", 0)
    if os.name == "nt":
        num_workers = 0

    train_loader = torch.utils.data.DataLoader(
        train_ds,
        batch_size=batch_size,
        num_workers=num_workers,
        shuffle=True,
        drop_last=True,
        pin_memory=True,
    )
    test_loader = torch.utils.data.DataLoader(
        test_ds,
        batch_size=batch_size,
        num_workers=num_workers,
        shuffle=False,
        drop_last=False,
        pin_memory=True,
    )

    return train_loader, test_loader


In [ ]:
%%writefile run_logging.py
from __future__ import annotations

import json
import logging
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

_logger = logging.getLogger("sparse_rebasin")
_metrics_file: Path | None = None

def _json_safe(value: Any) -> Any:

    if isinstance(value, dict):
        return {str(key): _json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [_json_safe(item) for item in value]
    if isinstance(value, (str, int, float, bool)) or value is None:
        return value
    if hasattr(value, "item"):
        try:
            return _json_safe(value.item())
        except (TypeError, ValueError):
            pass
    return str(value)

def start_run(name: str, config: dict[str, Any], run_dir: str | Path) -> None:

    global _metrics_file

    output_dir = Path(run_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    _metrics_file = output_dir / f"{name}_metrics.jsonl"

    if not _logger.handlers:
        handler = logging.StreamHandler()
        handler.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
        _logger.addHandler(handler)
    _logger.setLevel(logging.INFO)
    _logger.info("Started run %s", name)

    log_metrics({"event": "run_started", "run_name": name, "config": config})

def log_metrics(metrics: dict[str, Any]) -> None:

    if _metrics_file is None:
        raise RuntimeError("A run must be started before logging metrics")

    record = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        **metrics,
    }
    with _metrics_file.open("a", encoding="utf-8") as stream:
        stream.write(json.dumps(_json_safe(record)) + "\n")

def finish_run() -> None:

    if _metrics_file is not None:
        log_metrics({"event": "run_finished"})
        _logger.info("Finished run")


In [ ]:
%%writefile utils.py
import copy
import math
import os

import torch
import torch.nn as nn
import torch.nn.functional as F
import yaml
from loaders import cifar_dataloader
from models import *
from torch.cuda.amp import autocast
from torch.nn.utils.prune import custom_from_mask

def evaluate(model, dataloader, device):

    loss_fn = nn.CrossEntropyLoss()
    model.eval()
    correct = 0
    total = 0
    test_loss = 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            outputs = model(X)

            pred = outputs.argmax(dim=1)
            correct += (y == pred).sum().item()
            total += y.size(0)
            loss = loss_fn(outputs, y)
            test_loss += loss.item()

    test_accuracy = correct / total
    test_loss /= len(dataloader)
    return test_accuracy * 100, test_loss

def evaluate_top5(model, dataloader, device):

    loss_fn = nn.CrossEntropyLoss()
    model.eval()
    correct_top5 = 0
    total = 0
    test_loss = 0

    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            outputs = model(X)
            loss = loss_fn(outputs, y)
            test_loss += loss.item()

            _, pred_top5 = outputs.topk(5, dim=1)

            correct_top5 += sum([y[i] in pred_top5[i] for i in range(y.size(0))])
            total += y.size(0)

    test_accuracy_top5 = correct_top5 / total
    test_loss /= len(dataloader)

    return test_accuracy_top5 * 100, test_loss

def evaluate_merged_models(model_A_dense, permuted_model_b, alpha, config):
    device = config["device"]
    model_a = merge_models(model_A_dense, permuted_model_b, alpha, config)
    return model_a

def check_sparse_gradients(model):

    for name, param in model.named_parameters():
        if param.grad is not None:
            sparsity = float(torch.sum(param.grad == 0)) / torch.numel(param.grad)
            print(f"Gradient sparsity for {name}: {sparsity:.2f}")

def sparse_grad_tolerance(model, tolerance=0.0):

    for name, module in model.named_modules():
        if not hasattr(module, "weight_mask"):
            continue
        gradient = module.weight_orig.grad
        if gradient is None:
            continue
        masked = gradient[module.weight_mask == 0]
        if not torch.isfinite(masked).all() or (masked.abs() > tolerance).any():
            raise AssertionError(f"Nonzero or nonfinite masked gradients in {name}")

def calculate_mask_sparsity(weight_mask):
    total_params = torch.numel(weight_mask)
    zero_params = torch.sum(weight_mask == 0).item()
    sparsity = zero_params / total_params
    return sparsity, zero_params, total_params

def calculate_overall_sparsity_from_pth(model):
    total_zero_params = 0
    total_params = 0

    for name, param in model.state_dict().items():
        if 'weight_mask' in name:
            _, zero_params, num_params = calculate_mask_sparsity(param)
            total_zero_params += zero_params
            total_params += num_params

    overall_sparsity = total_zero_params / total_params if total_params > 0 else 0
    return overall_sparsity

def calculate_neuron_variances(model, unbiased=False):
    neuron_variances = {}
    with torch.no_grad():
        for name, param in model.named_parameters():
            if param.requires_grad and param.dim() > 1:
                variances = param.var(dim=list(range(1, param.dim())), unbiased=unbiased)
                neuron_variances[name] = variances
    return neuron_variances

def calculate_neuron_means(model):
    neuron_means = {}
    with torch.no_grad():
        for name, param in model.named_parameters():
            if param.requires_grad and param.dim() > 1:
                means = param.mean(dim=list(range(1, param.dim()))).detach()
                neuron_means[name] = means
    return neuron_means

def calculate_layer_variances(model, unbiased=False):
    variances = {}
    with torch.no_grad():
        for name, param in model.named_parameters():
            if param.requires_grad and param.dim() > 1:
                var = param.var(unbiased=unbiased).item()
                variances[name] = var
    return variances

def calculate_layer_means(model):
    means = {}
    with torch.no_grad():
        for name, param in model.named_parameters():
            if param.requires_grad and param.dim() > 1:
                mean = param.mean().item()
                means[name] = mean
    return means

def get_fan_in_tensor_mike(mask: torch.Tensor) -> torch.Tensor:
    if mask.dim() < 2:
        raise ValueError(
            "Fan in can not be computed for tensor with fewer than 2 dimensions"
        )
    if mask.dtype == torch.bool:
        fan_in_tensor = mask.sum(dim=list(range(1, mask.dim())))
    else:
        fan_in_tensor = (mask != 0.0).sum(dim=list(range(1, mask.dim())))
    return fan_in_tensor

def scale_model_with_fan_in(B_init_dense, A_sparse, config, mode: str = "fan_in", neuron_basis = True):
    if mode.lower() != "fan_in":
        raise NotImplementedError(
            "Only mode==`fan_in` has currently been implemented at this time."
        )

    if neuron_basis:
        layer_variances = calculate_neuron_variances(B_init_dense)
        layer_means = calculate_neuron_means(B_init_dense)

    else:
        layer_variances = calculate_layer_variances(B_init_dense)
        layer_means = calculate_layer_means(B_init_dense)

    print("print layer variance keys:", layer_variances.keys())
    print("print layer means keys:", layer_means.keys())

    for (name_B, m_B), (name_A, m_A) in zip(B_init_dense.named_modules(), A_sparse.named_modules()):
        if isinstance(m_B, (nn.Conv2d, nn.Linear)) and isinstance(m_A, (nn.Conv2d, nn.Linear)):
            with torch.no_grad():
                if config['model_type'] == 'ResNet':
                    if not hasattr(m_A, 'weight_mask'):
                        print(f"Skipping layer {name_A} ({m_A}) as it does not have weight_mask.")
                        continue
                mask = m_A.weight_mask
                if mask.shape != m_B.weight.shape:
                    raise ValueError(f"Sparsity mask and weight tensor shape do not match for layer {name_A}!")
                if 0 in m_B.weight.shape:
                    print(f"Initializing zero-element tensors is a no-op for layer {name_B}")
                    return m_B.weight
                fan_in = get_fan_in_tensor_mike(mask)

                weight_name = f"{name_B}.weight_orig"

                if weight_name not in layer_means or weight_name not in layer_variances:
                    print(f"Layer {name_B} not found in layer_means or layer_variances.")
                    continue

                mean_B = layer_means[weight_name]
                variance_B = layer_variances[weight_name]

                target_variance = 2 / fan_in
                if neuron_basis:
                    for i in range(m_B.weight.shape[0]):
                        if fan_in[i] != 0:

                            m_B.weight.data[i] -= mean_B[i]
                            sqrt_target_variance = math.sqrt(target_variance[i]) / math.sqrt(variance_B[i])
                            m_B.weight.data[i] *= sqrt_target_variance

                        elif fan_in[i] == 0:
                            m_B.weight.data[i] = 0
                else:
                    for i in range(m_B.weight.shape[0]):
                        if fan_in[i] != 0:

                            m_B.weight.data[i] -= mean_B
                            sqrt_target_variance = math.sqrt(target_variance[i]) / math.sqrt(variance_B)
                            m_B.weight.data[i] *= sqrt_target_variance

                        elif fan_in == 0:
                            m_B.weight.data[i] = 0

    return B_init_dense

def compute_loss_and_backward(model, data, target, loss_fn):
    model.train()
    output = model(data)
    loss = loss_fn(output, target)
    model.zero_grad()
    loss.backward()
    return loss

def calculate_gradient_norms(model):
    grad_norms = {}
    for name, module in model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            if module.weight.grad is not None:
                grad_norms[f"{name}.weight"] = module.weight.grad.norm().item()
    return grad_norms

def compute_gradient_norms(scaled_model, train_dl, config):

    for data, target in train_dl:
        data, target = data.to(config["device"]), target.to(config["device"])
        break

    loss_fn = nn.CrossEntropyLoss()
    compute_loss_and_backward(scaled_model, data, target, loss_fn)
    grad_norms = calculate_gradient_norms(scaled_model)

    print("Gradient norms after scaling and resetting BN stats:", grad_norms)

def transfer_sparsity_adnan(model_A, model_B):

    modules_to_replace = {}

    for (name_A, module_A), (name_B, module_B) in zip(model_A.features.named_modules(), model_B.features.named_modules()):
        assert(type(module_A) is type(module_B) and name_A == name_B)

        if name_A != 'classifier' and hasattr(module_A, "weight_mask"):
            print('Replacing layer in model B with masked layer:', name_A)
            modules_to_replace[name_A] = custom_from_mask(copy.deepcopy(module_B), name="weight", mask=module_A.weight_mask)

        else:
            print('Skipping layer in model A when copying masks:', name_A)

    print(modules_to_replace)
    for module_name, module in modules_to_replace.items():
        with torch.no_grad():
            assert(hasattr(model_B.features, module_name))
            setattr(model_B.features, module_name, module)
            assert(hasattr(getattr(model_B.features, module_name), 'weight_mask'))

    modules_to_replace = {}
    for (name_A, module_A), (name_B, module_B) in zip(model_A.classifier.named_modules(), model_B.classifier.named_modules()):
        assert(type(module_A) is type(module_B) and name_A == name_B)

        if hasattr(module_A, "weight"):
            print('Replacing layer in model B with masked layer:', name_A)
            modules_to_replace[name_A] = custom_from_mask(copy.deepcopy(module_B), name="weight", mask=module_A.weight_mask)
        else:
            print('Skipping layer in model A when copying masks:', name_A)

    print(modules_to_replace)
    for module_name, module in modules_to_replace.items():
        with torch.no_grad():
            setattr(model_B, 'classifier', module)
            assert(hasattr(getattr(model_B, 'classifier'), 'weight_mask'))

def check_hooks(model):
    def check_module_hooks(module):
        if isinstance(module, (torch.nn.Conv2d, torch.nn.Linear)) and hasattr(module, "weight_mask"):
            print(module, module._forward_pre_hooks)
            if len(module._forward_pre_hooks) == 0:
                print('Error')

        for child in module.children():
            check_module_hooks(child)

    check_module_hooks(model)

def get_model(config):
    if config["model_type"] == "VGG11":
        num_classes = dataset_num_classes.get(config['dataset'], 10)
        return VGG11_nofc(config, "VGG11", num_classes, init_weights=config["init_weights"]).to(config["device"])
    elif config["model_type"] == "ResNet":
        return ResNet(BasicBlock, [3, 3, 3], config, w=config["width_multiplier"]).to(config["device"])
    elif config["model_type"] == "ResNet50":
        return resnet50().to(config["device"])
    elif config["model_type"] == "LeNet":
        return LeNet(
            config,
            w=config.get("width_multiplier", 1),
            init_weights=config.get("init_weights", True),
        ).to(config["device"])
    else:
        raise ValueError(f"Unsupported model type: {config['model_type']}")

def merge_models(m0, m1, alpha, config):
    net = get_model(config)
    sd0, sd1 = m0.state_dict(), m1.state_dict()
    sd_alpha = {
        k: (1 - alpha) * sd0[k].to(config['device']) + alpha * sd1[k].to(config['device'])
        for k in sd0.keys()
        if k in sd1
    }
    net.load_state_dict(sd_alpha, strict=False)
    return net

"""
The following function is taken from REPAIR.
"""

def reset_bn_stats(model, loader):
    device = next(model.parameters()).device

    for m in model.modules():
        if type(m) == nn.BatchNorm2d:
            m.momentum = None
            m.reset_running_stats()

    model.train()
    with torch.no_grad(), autocast(enabled=device.type == "cuda"):
        for images, _ in loader:
            output = model(images.to(device))

def reset_bn_stats_sparse(model, loader, iter = 40, reset_params=False):

    for m in model.modules():
        if type(m) == nn.BatchNorm2d:
            m.momentum = 0.1
            m.reset_running_stats()
            if reset_params == True:
                m.reset_parameters()

    model.train()
    with torch.no_grad():
        for epoch in range(iter):
            for images, _ in loader:
                output = model(images.cuda())

def compare_model_weights(model1, model2):
    for (name1, param1), (name2, param2) in zip(model1.named_parameters(), model2.named_parameters()):
        if "bias" not in name1 and "bias" not in name2:
            if not torch.equal(param1, param2):
                print(f"Difference found in layer: {name1}")
                return False
    print("All weights match for all layers.")
    return True

def transfer_sparsity_resnet(model_A, model_B):

    modules_to_replace = {}

    for (name_A, module_A), (name_B, module_B) in zip(model_A.named_modules(), model_B.named_modules()):
        if type(module_A) is not type(module_B) or name_A != name_B:
            print(f"Mismatch found:")
            print(f"- model_A: {name_A}, type: {type(module_A)}")
            print(f"- model_B: {name_B}, type: {type(module_B)}")
        assert(type(module_A) is type(module_B) and name_A == name_B)

        if hasattr(module_A, "weight_mask"):
            print('Replacing layer in model B with masked layer:', name_A)
            modules_to_replace[name_A] = custom_from_mask(copy.deepcopy(module_B), name="weight", mask=module_A.weight_mask)
        else:
            print('Skipping layer in model A when copying masks:', name_A)

    for module_name, module in modules_to_replace.items():
        with torch.no_grad():
            mod_attr = model_B
            for i in range(len(module_name.split('.'))-1):
                if len(module_name.split('.')[i])!=1:
                    mod_attr = getattr(model_B, module_name.split('.')[i])
                else:
                    mod_attr = mod_attr[int(module_name.split('.')[i])]

            assert(hasattr(mod_attr, module_name.split('.')[-1]))
            setattr(mod_attr, module_name.split('.')[-1], module)
            assert(hasattr(getattr(mod_attr, module_name.split('.')[-1]), 'weight_mask'))

class TrackLayer(nn.Module):
    def __init__(self, layer, one_d=False):
        super().__init__()
        self.layer = layer
        dim = layer.conv3.out_channels
        self.bn = nn.BatchNorm2d(dim)

    def get_stats(self):
        return (self.bn.running_mean, self.bn.running_var.sqrt())

    def forward(self, x):
        x1 = self.layer(x)
        self.bn(x1)
        return x1

class ResetLayer(nn.Module):
    def __init__(self, layer, one_d=False):
        super().__init__()
        self.layer = layer
        dim = layer.conv3.out_channels
        self.bn = nn.BatchNorm2d(dim)

    def set_stats(self, goal_mean, goal_std):
        self.bn.bias.data = goal_mean
        self.bn.weight.data = goal_std

    def forward(self, x):
        x1 = self.layer(x)
        return self.bn(x1)

def make_tracked_net(net, config):
    net1 = get_model(config)
    net1.load_state_dict(net.state_dict())
    for i in range(4):
        layer = getattr(net1, 'layer%d' % (i+1))
        for j, block in enumerate(layer):
            layer[j] = TrackLayer(block).cuda()
    return net1

def make_repaired_net(net, config):
    net1 = get_model(config)
    net1.load_state_dict(net.state_dict())
    for i in range(4):
        layer = getattr(net1, 'layer%d' % (i+1))
        for j, block in enumerate(layer):
            layer[j] = ResetLayer(block).cuda()
    return net1

def repair_merged_model(permuted_model_0, model_1, model_a, train_dl, config, alpha=0.5):

    model0 = evaluate_merged_models(permuted_model_0, model_1, 0, config)
    model1 = evaluate_merged_models(permuted_model_0, model_1, 1, config)

    wrap0 = make_tracked_net(model0)
    wrap1 = make_tracked_net(model1)
    reset_bn_stats(wrap0, train_dl)
    reset_bn_stats(wrap1, train_dl)

    wrap_a = make_repaired_net(model_a)

    for track0, track1, reset_a in zip(wrap0.modules(), wrap1.modules(), wrap_a.modules()):
        if not isinstance(track0, TrackLayer):
            continue
        assert (isinstance(track0, TrackLayer)
                and isinstance(track1, TrackLayer)
                and isinstance(reset_a, ResetLayer))

        mu0, std0 = track0.get_stats()
        mu1, std1 = track1.get_stats()

        goal_mean = (1 - alpha) * mu0 + alpha * mu1
        goal_std = (1 - alpha) * std0 + alpha * std1
        reset_a.set_stats(goal_mean, goal_std)

    reset_bn_stats(wrap_a, train_dl)

    return wrap_a

def update_config_with_slurm_tmpdir(config, slurm_tmpdir):
    config['dataset_root']['imagenet'] = os.path.join(slurm_tmpdir, 'imagenet')
    return config


In [ ]:
%%writefile training.py
import torch
import torch.nn as nn
import torch.optim as optim
import math
from loaders import cifar_dataloader
from utils import evaluate, sparse_grad_tolerance
from run_logging import log_metrics
import yaml

def train_epoch(model, dataloader, optimizer, device, sparse=False):

    model.train()
    running_loss = 0
    loss_fn = nn.CrossEntropyLoss()

    for batch_index, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()

        pred = model(X)
        loss = loss_fn(pred, y)
        loss_value = loss.item()
        if not math.isfinite(loss_value):
            raise FloatingPointError("Nonfinite training loss; stop before producing a misleading final result")
        running_loss += loss_value

        loss.backward()

        if sparse and batch_index == 0:
            sparse_grad_tolerance(model)
        optimizer.step()

    return running_loss / len(dataloader)

def trainer(model, optimizer_config, epochs, batch_size, device, model_name, config, sparse=False, training_type="naive", k=0):

    comparison = config.get("comparison", {})
    save_rewind = comparison.get("enabled", False) and model_name in ("Model_A_Dense", "Model_B_Dense")
    rewind_epoch = int(comparison.get("rewind_epoch", 0))
    if save_rewind:
        if not 0 <= rewind_epoch <= epochs:
            raise ValueError(f"{model_name}: rewind epoch {rewind_epoch} exceeds dense training epochs {epochs}")
        if rewind_epoch == 0:
            torch.save(model.state_dict(), f"{config['save_path']}{model_name}_rewind_0_seed_{config['seed']}.pth")

    optimizer_type = getattr(optim, optimizer_config["type"])
    lr = optimizer_config["sparse_lr"] if sparse else optimizer_config["lr"]
    optimizer = optimizer_type(
        model.parameters(),
        lr=lr,
        momentum=optimizer_config["momentum"],
        weight_decay=optimizer_config["weight_decay"],
    )
    lrs = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs, eta_min=0.0)
    train_dl, test_dl = cifar_dataloader(batch_size, config)

    for epoch in range(epochs):
        train_loss_est = train_epoch(model, train_dl, optimizer, device, sparse=sparse)
        if save_rewind and epoch + 1 == rewind_epoch:
            torch.save(model.state_dict(), f"{config['save_path']}{model_name}_rewind_{rewind_epoch}_seed_{config['seed']}.pth")
        if not sparse and config.get("save_legacy_checkpoints", True) and epoch % 5 == 0:
            torch.save(model.state_dict(), config['save_path'] + model_name + '_sparsity_' + str(config['pruning']['sparsity'])+ '_seed_' + str(config['seed']) + '_epoch_' + str(epoch))
        test_accu, test_loss = evaluate(model, test_dl, device)
        print(
            f"Epoch: {epoch+1:2d} - Test accuracy: {test_accu:.2f} - Test loss: {test_loss:.4f} - ",
            f"Train loss est.: {train_loss_est:.4f} - Learning rate: {optimizer.param_groups[0]['lr']:.4f}",
        )
        log_metrics({
            "event": "epoch_summary", "model": model_name, "epoch": epoch + 1,
            "train_loss": train_loss_est, "test_loss": test_loss,
            "test_accuracy": test_accu, "learning_rate": optimizer.param_groups[0]["lr"],
        })
        if sparse:
            if training_type == "naive":
                log_metrics(
                    {
                        f"{model_name} Naive Epoch": epoch + 1,
                        f"{model_name} Naive Train Loss": train_loss_est,
                        f"{model_name} Naive Test Loss": test_loss,
                        f"{model_name} Naive Test Accuracy": test_accu,
                    }
                )
            elif training_type == "permuted":
                log_metrics(
                    {
                        f"{model_name}  Epoch": epoch + 1,
                        f"{model_name}  Train Loss": train_loss_est,
                        f"{model_name}  Test Loss": test_loss,
                        f"{model_name}  Test Accuracy": test_accu,
                    }
                )
            elif training_type == "LTH":
                log_metrics(
                    {
                        f"{model_name}  Epoch": epoch + 1,
                        f"{model_name}  Train Loss": train_loss_est,
                        f"{model_name}  Test Loss": test_loss,
                        f"{model_name}  Test Accuracy": test_accu,
                    }
                )
            elif training_type == "progressive":
                log_metrics(
                    {
                        f"{model_name} Progressive Epoch": epoch + 1,
                        f"{model_name} Progressive Train Loss": train_loss_est,
                        f"{model_name} Progressive Test Loss": test_loss,
                        f"{model_name} Progressive Test Accuracy": test_accu,
                    }
                )
        else:
            log_metrics(
                {
                    f"{model_name}  Epoch": epoch + 1,
                    f"{model_name}  Train Loss": train_loss_est,
                    f"{model_name}  Test Loss": test_loss,
                    f"{model_name}  Test Accuracy": test_accu,
                }
            )
        lrs.step()


In [ ]:
%%writefile pruning.py
import torch
import os
import torch.nn.utils.prune as prune
import torch.optim as optim
from training import train_epoch
from loaders import cifar_dataloader
from utils import evaluate, calculate_overall_sparsity_from_pth, check_sparse_gradients, calculate_mask_sparsity, get_model
from run_logging import log_metrics
import yaml

def prune_model(
    config,
    model,
    target_sparsity,
    optimizer_config,
    prune_epochs,
    initial_lr,
    batch_size,
    device,
    initial_prune_perc=0.2,
    train_epochs_per_prune=5,
    sparsity_tolerance=0.005,
):
    total_epochs = 0
    train_dl, test_dl = cifar_dataloader(batch_size, config)

    if config["model_type"] == "VGG11":
        parameters_to_prune = [
            (module, 'weight') for name, module in model.named_modules()
            if isinstance(module, (torch.nn.Conv2d, torch.nn.Linear))
        ]
    elif config["model_type"] == "ResNet":
        parameters_to_prune = [
            (module, 'weight') for name, module in model.named_modules()
            if isinstance(module, (torch.nn.Conv2d, torch.nn.Linear)) and 'shortcut' not in name
        ]
    elif config["model_type"] == "ResNet50":
        parameters_to_prune = [
            (module, 'weight') for name, module in model.named_modules()
            if isinstance(module, (torch.nn.Conv2d, torch.nn.Linear)) and 'shortcut' not in name and 'downsample' not in name
        ]
    elif config["model_type"] == "LeNet":
        parameters_to_prune = [
            (module, 'weight') for module in model.modules()
            if isinstance(module, (torch.nn.Conv2d, torch.nn.Linear))
        ]
    else:
        raise ValueError(f"Unsupported model type: {config['model_type']}")

    if config.get("comparison", {}).get("enabled", False):

        total_weights = sum(module.weight.numel() for module, _ in parameters_to_prune)
        sparsity_tolerance = 0.5 / total_weights

    current_sparsity = calculate_overall_sparsity_from_pth(model)
    print(f"Initial sparsity: {current_sparsity*100:.2f}%")

    for epoch in range(prune_epochs):
        optimizer_type = getattr(optim, optimizer_config["type"])
        optimizer = optimizer_type(
            model.parameters(), lr=initial_lr, **optimizer_config.get('params', {})
        )
        lrs = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, train_epochs_per_prune, eta_min=0.0
        )

        current_sparsity = calculate_overall_sparsity_from_pth(model)
        print(f"Current sparsity before pruning: {current_sparsity*100:.2f}%")

        if current_sparsity < target_sparsity:
            max_additional_sparsity = target_sparsity - current_sparsity
            adjusted_prune_perc = min(
                initial_prune_perc, max_additional_sparsity / (1 - current_sparsity)
            )
        else:
            adjusted_prune_perc = 0

        print(f"Adjusted prune percentage: {adjusted_prune_perc*100:.2f}%")

        prune.global_unstructured(
            parameters_to_prune,
            pruning_method=prune.L1Unstructured,
            amount=adjusted_prune_perc
        )

        for epoch_num in range(train_epochs_per_prune):
            train_loss_est = train_epoch(model, train_dl, optimizer, device)
            test_accu, test_loss = evaluate(model, test_dl, device)
            print(
                f"Prune Epoch: {epoch:2d}, Train Epoch: {epoch_num} - "
                f"Test accuracy: {test_accu:.2f} - Test loss: {test_loss:.4f} - "
                f"Train loss est.: {train_loss_est:.4f} - Learning rate: {optimizer.param_groups[0]['lr']:.4f}"
            )
            log_metrics(
                {
                    "Model A Sparse Prune Epoch": epoch + 1,
                    "Model A Sparse Train Loss": train_loss_est,
                    "Model A Sparse Test Loss": test_loss,
                    "Model A Sparse Test Accuracy": test_accu,
                }
            )
            total_epochs += 1
            lrs.step()

        overall_sparsity = calculate_overall_sparsity_from_pth(model)
        print(f"Prune Epoch: {epoch} - Overall Sparsity: {overall_sparsity*100:.2f}%")

        for name, param in model.state_dict().items():
            if 'weight_mask' in name:
                sparsity, zero_params, total_params = calculate_mask_sparsity(param)
                print(f"Layer {name} - Sparsity: {sparsity*100:.2f}% - Zero params: {zero_params} - Total params: {total_params}")

        check_sparse_gradients(model)

        if overall_sparsity >= target_sparsity - sparsity_tolerance:
            print(f"Target sparsity of {target_sparsity*100:.2f}% reached. Terminating pruning.")
            break
    return total_epochs


In [ ]:
%%writefile progressive.py
from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.utils.prune as prune

def eligible_modules(model: nn.Module, model_type: str) -> list[tuple[str, nn.Module]]:

    excluded = ()
    if model_type == "ResNet":
        excluded = ("shortcut",)
    elif model_type == "ResNet50":
        excluded = ("shortcut", "downsample")
    elif model_type not in {"VGG11", "LeNet"}:
        raise ValueError(f"Unsupported model type: {model_type}")

    return [
        (name, module)
        for name, module in model.named_modules()
        if isinstance(module, (nn.Conv2d, nn.Linear))
        and not any(part in name for part in excluded)
    ]

def _mask(module: nn.Module) -> torch.Tensor:
    if not hasattr(module, "weight_mask"):
        raise ValueError("All progressive layers must have a weight_mask")
    return module.weight_mask.detach()

def mask_statistics(model: nn.Module, model_type: str) -> tuple[int, int]:

    active = total = 0
    for _, module in eligible_modules(model, model_type):
        mask = _mask(module)
        active += int(mask.count_nonzero().item())
        total += mask.numel()
    return active, total

def snapshot_masks(model: nn.Module, model_type: str) -> dict[str, torch.Tensor]:

    return {
        name: _mask(module).bool().clone()
        for name, module in eligible_modules(model, model_type)
    }

def mask_retention_statistics(
    model: nn.Module,
    model_type: str,
    initial_masks: dict[str, torch.Tensor],
    grown_masks: dict[str, torch.Tensor],
) -> dict[str, int | float]:

    final_masks = snapshot_masks(model, model_type)
    names = {name for name, _ in eligible_modules(model, model_type)}
    if set(initial_masks) != names or set(grown_masks) != names:
        raise ValueError("Mask snapshots do not match the model's eligible layers")

    initial = torch.cat([initial_masks[name].reshape(-1) for name in sorted(names)])
    grown = torch.cat([grown_masks[name].reshape(-1) for name in sorted(names)])
    final = torch.cat([final_masks[name].reshape(-1) for name in sorted(names)])
    added = grown & ~initial

    initial_count = int(initial.sum().item())
    added_count = int(added.sum().item())
    return {
        "initial_active": initial_count,
        "added_active": added_count,
        "final_active": int(final.sum().item()),
        "initial_retained": int((initial & final).sum().item()),
        "added_retained": int((added & final).sum().item()),
        "initial_retention": (
            float((initial & final).sum().item() / initial_count)
            if initial_count
            else 1.0
        ),
        "added_retention": (
            float((added & final).sum().item() / added_count) if added_count else 1.0
        ),
    }

def validate_mask_compatibility(
    source_model: nn.Module, target_model: nn.Module, model_type: str
) -> None:
    source_layers = eligible_modules(source_model, model_type)
    target_layers = eligible_modules(target_model, model_type)
    if [name for name, _ in source_layers] != [name for name, _ in target_layers]:
        raise ValueError("Source and target progressive layers do not match")
    for (name_a, source), (name_b, target) in zip(source_layers, target_layers):
        if source.weight.shape != target.weight.shape:
            raise ValueError(
                f"Layer shape mismatch for {name_a}/{name_b}: "
                f"{tuple(source.weight.shape)} != {tuple(target.weight.shape)}"
            )

def _install_masks(
    model: nn.Module, masks: dict[str, torch.Tensor], model_type: str
) -> None:

    for name, module in eligible_modules(model, model_type):
        if hasattr(module, "weight_mask"):
            module.weight_mask.data.copy_(
                masks[name].to(
                    device=module.weight_mask.device, dtype=module.weight_mask.dtype
                )
            )
        else:
            prune.custom_from_mask(
                module, "weight", masks[name].to(device=module.weight.device)
            )

def transfer_mask(
    source_model: nn.Module, target_model: nn.Module, model_type: str
) -> nn.Module:

    validate_mask_compatibility(source_model, target_model, model_type)
    masks = {
        name: _mask(source).to(dtype=target.weight.dtype).clone()
        for (name, source), (_, target) in zip(
            eligible_modules(source_model, model_type),
            eligible_modules(target_model, model_type),
        )
    }
    _install_masks(target_model, masks, model_type)
    return target_model

def grow_mask(
    model: nn.Module,
    model_type: str,
    target_density: float,
    selection: str = "magnitude",
    generator: torch.Generator | None = None,
) -> tuple[int, int]:

    if not 0.0 <= target_density <= 1.0:
        raise ValueError("target_density must be between 0 and 1")
    if selection not in {"magnitude", "random", "gradient"}:
        raise ValueError("selection must be 'magnitude', 'random', or 'gradient'")

    layers = eligible_modules(model, model_type)
    if selection == "gradient":
        for _, module in layers:
            weights = getattr(module, "weight_orig", module.weight)
            if weights.grad is None:
                raise ValueError(
                    "gradient selection requires gradients on all progressive layers"
                )

    old_active, total = mask_statistics(model, model_type)
    target_active = int(round(target_density * total))
    if target_active < old_active:
        raise ValueError(
            f"Progressive growth cannot reduce active weights: "
            f"{old_active} -> {target_active}"
        )
    add_count = target_active - old_active
    if add_count == 0:
        return old_active, total

    candidate_indices: list[torch.Tensor] = []
    candidate_scores: list[torch.Tensor] = []
    for name, module in layers:
        mask = _mask(module).bool()
        weights = getattr(module, "weight_orig", module.weight)
        if selection == "gradient":
            scores = weights.grad.detach().abs()
        else:
            scores = weights.detach().abs()
        inactive = ~mask.reshape(-1)
        candidate_indices.append(inactive.nonzero(as_tuple=False).flatten())
        candidate_scores.append(scores.reshape(-1)[inactive])

    candidate_count = sum(indices.numel() for indices in candidate_indices)
    if add_count > candidate_count:
        raise ValueError("Requested density exceeds available inactive weights")

    scores = torch.cat(candidate_scores)
    if selection == "magnitude":
        _, selected = torch.topk(scores, add_count, largest=True, sorted=False)
    elif selection == "random":
        order = torch.randperm(candidate_count, generator=generator)
        selected = order[:add_count].to(device=scores.device)
    else:
        _, selected = torch.topk(scores, add_count, largest=True, sorted=False)

    masks = {name: _mask(module).clone() for name, module in layers}
    offsets = torch.tensor(
        [
            0,
            *torch.cumsum(
                torch.tensor(
                    [indices.numel() for indices in candidate_indices],
                    dtype=torch.long,
                ),
                dim=0,
            ).tolist(),
        ],
        device=scores.device,
    )

    layer_indices = torch.bucketize(selected, offsets[1:-1], right=True)
    for layer_index, (name, _) in enumerate(layers):
        selected_in_layer = selected[layer_indices == layer_index]
        if selected_in_layer.numel() == 0:
            continue
        local_indices = selected_in_layer - offsets[layer_index]
        flat_mask = masks[name].reshape(-1)
        flat_mask[candidate_indices[layer_index][local_indices]] = 1
    _install_masks(model, masks, model_type)
    return mask_statistics(model, model_type)

def assert_density(
    model: nn.Module, model_type: str, expected_density: float, tolerance: float = 1e-6
) -> None:
    active, total = mask_statistics(model, model_type)
    actual = active / total

    tolerance = max(tolerance, 0.5 / total + 1e-12)
    if abs(actual - expected_density) > tolerance:
        raise AssertionError(
            f"Expected density {expected_density:.12f}, got {actual:.12f}"
        )


In [ ]:
%%writefile comparison.py
from __future__ import annotations

import hashlib
import json
import random
from pathlib import Path

import numpy as np
import torch

from loaders import cifar_dataloader
from progressive import eligible_modules, mask_statistics, transfer_mask
from run_logging import finish_run, log_metrics, start_run
from training import trainer
from utils import evaluate, get_model

METHOD_LABELS = {
    "naive": "LTH Problem (Naive)",
    "permuted": "Our Solution (Permuted)",
    "lth": "LTH Solution",
    "progressive": "Progressive (pruned + trained)",
}

def seed_training(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def initialize_dense_model(config, role):

    if role not in ("A", "B"):
        raise ValueError("Dense model role must be A or B")
    offset = 0 if role == "A" else 1_000_003
    seed_training(int(config["seed"]) + offset)
    return get_model(config)

def comparison_settings(config):
    settings = config.get("comparison", {})
    terminal_epoch = int(settings.get("total_epochs", config["sparse_epochs"]))
    rewind = int(settings.get("rewind_epoch", 0))
    minimum_post_prune = int(settings.get("min_post_prune_epochs", 1))
    if not 0 <= rewind < terminal_epoch:
        raise ValueError("Comparison requires 0 <= rewind_epoch < total_epochs (T)")
    if minimum_post_prune < 1:
        raise ValueError("min_post_prune_epochs must be at least 1")
    return terminal_epoch, rewind, minimum_post_prune

def rewind_path(config, role):
    _, rewind, _ = comparison_settings(config)
    return Path(config["save_path"]) / (
        f"Model_{role}_Dense_rewind_{rewind}_seed_{config['seed']}.pth"
    )

def load_rewind_model(config, role):
    path = rewind_path(config, role)
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing exact rewind checkpoint: {path}. Run pretraining with "
            "comparison.enabled=true. Legacy epoch_0 is AFTER one epoch "
            "and cannot be used as initialization (k=0)."
        )
    model = get_model(config)
    model.load_state_dict(torch.load(path, map_location=config["device"], weights_only=True))
    return model

def target_active_count(total, sparsity):
    return total - int(round(total * sparsity))

def pruning_rounds_required(active, total, sparsity, fraction):

    target = target_active_count(total, sparsity)
    if not 0 < fraction < 1 or active < target:
        raise ValueError("Invalid pruning fraction or a source mask below target density")
    rounds = 0
    while active > target:
        removed = min(int(round(fraction * active)), active - target)
        if removed <= 0:
            raise ValueError("Pruning fraction is too small to remove another weight")
        active -= removed
        rounds += 1
    return rounds

def growth_stage_epochs(config):

    growth = config["progressive"]
    targets = growth.get("target_densities", [])
    if not targets:
        raise ValueError("progressive.target_densities must contain at least one target")
    epochs = growth.get("epochs_per_stage", 1)
    if isinstance(epochs, int) and not isinstance(epochs, bool):
        result = [epochs] * len(targets)
    elif isinstance(epochs, (list, tuple)):
        result = list(epochs)
    else:
        raise ValueError("epochs_per_stage must be an integer or a list of integers")
    if len(result) != len(targets):
        raise ValueError("epochs_per_stage must have one entry per target density")
    if any(not isinstance(value, int) or isinstance(value, bool) or value < 1 for value in result):
        raise ValueError("Each growth stage must train for a positive integer number of epochs")
    return result

def progressive_budget(config, total_weights):
    terminal, rewind, minimum_post_prune = comparison_settings(config)
    growth = config["progressive"]
    growth_epochs = sum(growth_stage_epochs(config))
    rounds = pruning_rounds_required(
        int(round(growth["target_densities"][-1] * total_weights)),
        total_weights,
        config["pruning"]["sparsity"],
        config["pruning"]["prune_perc"],
    )
    if rounds > config["pruning"]["prune_epochs"]:
        raise ValueError(f"Progressive comparison needs at least {rounds} pruning rounds")
    pruning_epochs = rounds * int(growth.get(
        "train_epochs_per_prune", config["pruning"]["train_epochs_per_prune"]
    ))
    minimum_terminal = rewind + growth_epochs + pruning_epochs + minimum_post_prune
    if terminal < minimum_terminal:
        raise ValueError(
            f"T={terminal} is too small. Use T >= {minimum_terminal}: "
            f"k={rewind}, growth={growth_epochs}, pruning={pruning_epochs}, "
            f"post-pruning >= {minimum_post_prune}."
        )
    return growth_epochs, pruning_epochs

def configuration_signature(config):

    settings = {key: value for key, value in config.items() if key != "save_path"}
    return hashlib.sha256(json.dumps(settings, sort_keys=True).encode()).hexdigest()

def record_initial_diagnostics(method, model, config):

    layers = []
    for name, module in eligible_modules(model, config["model_type"]):
        weights = module.weight_orig.detach()
        mask = module.weight_mask.detach()
        effective = weights * mask
        layers.append({
            "layer": name, "active_weights": int(mask.count_nonzero().item()),
            "total_weights": mask.numel(),
            "density": float(mask.count_nonzero().item() / mask.numel()),
            "dense_weight_rms": float(weights.square().mean().sqrt().item()),
            "masked_weight_rms": float(effective.square().mean().sqrt().item()),
        })
    _, rewind, _ = comparison_settings(config)
    record = {
        "event": "initial_diagnostics", "method": method,
        "completed_dense_epochs_at_rewind": rewind,
        "legacy_epoch_label_for_same_checkpoint": rewind - 1 if rewind > 0 else None,
        "initialization_checkpoint": str(rewind_path(config, "A" if method == "lth" else "B")),
        "initialization_seed": int(config["seed"]) + (0 if method == "lth" else 1_000_003),
        "layers": layers,
    }
    output = Path(config["save_path"]) / "diagnostics"
    output.mkdir(parents=True, exist_ok=True)
    (output / f"{method}_initial_seed_{config['seed']}.json").write_text(
        json.dumps(record, indent=2), encoding="utf-8"
    )
    log_metrics(record)
    return record

def record_final_result(method, model, config, stages, test_loader=None):

    terminal, rewind, minimum_post_prune = comparison_settings(config)
    if rewind + sum(stages.values()) != terminal:
        raise ValueError(f"{method}: completed epochs do not reach T={terminal}")
    if method == "progressive" and stages.get("post_pruning", 0) < minimum_post_prune:
        raise ValueError("Progressive result requires training after the last pruning step")
    active, total = mask_statistics(model, config["model_type"])
    expected = target_active_count(total, config["pruning"]["sparsity"])
    if active != expected:
        raise ValueError(f"{method}: final mask has {active} active weights, expected {expected}")
    if test_loader is None:
        _, test_loader = cifar_dataloader(config["batch_size"], config)
    accuracy, loss = evaluate(model, test_loader, config["device"])
    output = Path(config["save_path"]) / "final_results"
    output.mkdir(parents=True, exist_ok=True)
    checkpoint = output / f"{method}_seed_{config['seed']}.pth"
    torch.save(model.state_dict(), checkpoint)
    record = {
        "method": method,
        "label": METHOD_LABELS[method],
        "seed": config["seed"],
        "terminal_epoch": terminal,
        "rewind_epoch": rewind,
        "checkpoint_convention": "completed_dense_epochs",
        "legacy_epoch_label_for_same_checkpoint": rewind - 1 if rewind > 0 else None,
        "trained_epochs": sum(stages.values()),
        "stage_epochs": stages,
        "test_accuracy": float(accuracy),
        "test_loss": float(loss),
        "target_sparsity": config["pruning"]["sparsity"],
        "actual_sparsity": 1 - active / total,
        "active_weights": active,
        "total_weights": total,
        "initialization_checkpoint": str(rewind_path(config, "A" if method == "lth" else "B")),
        "checkpoint": str(checkpoint),
        "configuration_signature": configuration_signature(config),
    }
    result_path = output / f"{method}_seed_{config['seed']}.json"
    temporary = result_path.with_suffix(".json.tmp")
    temporary.write_text(json.dumps(record, indent=2), encoding="utf-8")
    temporary.replace(result_path)
    log_metrics({"event": "final_result", **record})
    print(f"FINAL {METHOD_LABELS[method]}: t=T={terminal}, accuracy={accuracy:.2f}%, loss={loss:.4f}")
    return record

def train_sparse_baselines(seed, config):

    terminal, rewind, _ = comparison_settings(config)
    output = Path(config["save_path"])
    suffix = f"sparsity_{config['pruning']['sparsity']}_seed_{seed}"
    source_path = output / f"model_a_sparse_{suffix}"
    permuted_path = output / f"permuted_model_a_sparse_{suffix}"

    for path in (source_path, permuted_path, rewind_path(config, "A"), rewind_path(config, "B")):
        if not path.is_file():
            raise FileNotFoundError(f"Missing checkpoint {path}; rerun comparison-enabled pretraining")
    records = []
    for method, role, mask_path, training_type in (
        ("naive", "B", source_path, "naive"),
        ("permuted", "B", permuted_path, "permuted"),
        ("lth", "A", source_path, "LTH"),
    ):
        model = load_rewind_model(config, role)
        source = torch.load(mask_path, map_location=config["device"], weights_only=False)
        transfer_mask(source, model, config["model_type"])
        del source
        active, total = mask_statistics(model, config["model_type"])
        if active != target_active_count(total, config["pruning"]["sparsity"]):
            raise ValueError(f"{method}: source mask has incorrect sparsity; rerun pretraining")

        seed_training(seed)
        start_run(f"{method}_{config['model_type']}_seed_{seed}", {
            "method": method, "T": terminal, "k": rewind,
        }, config["save_path"])
        record_initial_diagnostics(method, model, config)
        trainer(
            model, config["optimizer"], terminal - rewind, config["batch_size"],
            config["device"], f"Comparison_{method}", config,
            sparse=True, training_type=training_type,
        )
        records.append(record_final_result(method, model, config, {"sparse_training": terminal - rewind}))
        finish_run()
        del model
    return records


In [ ]:
%%writefile comparison_plot.py
import csv
import json
import math
from pathlib import Path

from comparison import METHOD_LABELS, comparison_settings, configuration_signature, target_active_count

def load_final_results(run_dir, config):
    terminal, rewind, minimum_post_prune = comparison_settings(config)
    signature = configuration_signature(config)
    results = []
    for method in METHOD_LABELS:
        path = Path(run_dir) / "final_results" / f"{method}_seed_{config['seed']}.json"
        if not path.is_file():
            raise FileNotFoundError(f"Missing final result for {METHOD_LABELS[method]}: {path}")
        record = json.loads(path.read_text(encoding="utf-8"))
        if (record["method"] != method or record["seed"] != config["seed"]
                or record["configuration_signature"] != signature
                or record["terminal_epoch"] != terminal or record["rewind_epoch"] != rewind
                or record["trained_epochs"] != terminal - rewind
                or sum(record["stage_epochs"].values()) != record["trained_epochs"]):
            raise ValueError(f"{method}: incompatible or incomplete result; rerun with the current settings")
        if record["target_sparsity"] != config["pruning"]["sparsity"]:
            raise ValueError(f"{method}: different target sparsity")
        if record["active_weights"] != target_active_count(record["total_weights"], record["target_sparsity"]):
            raise ValueError(f"{method}: final mask is not at target sparsity")
        if not math.isfinite(record["test_accuracy"]) or not 0 <= record["test_accuracy"] <= 100:
            raise ValueError(f"{method}: invalid final test accuracy")
        if not math.isfinite(record["test_loss"]):
            raise ValueError(f"{method}: invalid final test loss")
        if method == "progressive" and record["stage_epochs"].get("post_pruning", 0) < minimum_post_prune:
            raise ValueError("Progressive result is missing post-pruning training")
        results.append(record)
    if len({(record["total_weights"], record["active_weights"]) for record in results}) != 1:
        raise ValueError("The four results have different model or mask sizes")
    return results

def plot_final_comparison(run_dir, config, output_dir):
    import matplotlib.pyplot as plt

    results = load_final_results(run_dir, config)
    terminal, rewind, _ = comparison_settings(config)
    output_dir = Path(output_dir)
    columns = ["method", "label", "terminal_epoch", "rewind_epoch", "trained_epochs",
               "test_accuracy", "test_loss", "actual_sparsity", "active_weights", "total_weights"]
    with (output_dir / "final_comparison.csv").open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(results)
    (output_dir / "final_comparison.json").write_text(json.dumps(results, indent=2), encoding="utf-8")
    labels = ["LTH Problem\n(Naive)", "Our Solution\n(Permuted)", "LTH Solution",
              "Progressive\n(pruned + trained)"]
    accuracy = [record["test_accuracy"] for record in results]
    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(labels, accuracy, color=["#e69f00", "#0072b2", "#009e73", "#cc79a7"], width=0.65)
    ax.bar_label(bars, labels=[f"{value:.2f}%" for value in accuracy], padding=5)
    ax.set_ylim(0, 100)
    ax.set_ylabel("Final test accuracy (%)")
    ax.set_title(
        f"{config['model_type']} / {config['dataset']} — final results at t=T={terminal}\n"
        f"seed={config['seed']}, k={rewind}, sparsity={100 * config['pruning']['sparsity']:.0f}%"
    )
    ax.set_axisbelow(True)
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    fig.savefig(output_dir / "final_comparison.png", dpi=180)
    fig.savefig(output_dir / "final_comparison.pdf")
    print(f"Final results at t=T={terminal} (k={rewind}):")
    for record in results:
        print(f"{record['label']}: accuracy={record['test_accuracy']:.2f}%, "
              f"loss={record['test_loss']:.4f}, sparsity={100 * record['actual_sparsity']:.6f}%, "
              f"training stages={record['stage_epochs']}")
    return fig, results

def plot_baseline_learning_curves(run_dir, config, output_dir):

    import matplotlib.pyplot as plt

    terminal, rewind, _ = comparison_settings(config)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    rows = []
    for method, color in (("naive", "#e69f00"), ("permuted", "#0072b2"), ("lth", "#009e73")):
        path = Path(run_dir) / f"{method}_{config['model_type']}_seed_{config['seed']}_metrics.jsonl"
        if not path.exists():
            continue

        epochs = {}
        for line in path.read_text(encoding="utf-8").splitlines():
            record = json.loads(line)
            if record.get("event") == "epoch_summary":
                epochs[record["epoch"]] = record
        points = [epochs[key] for key in sorted(epochs)]
        x = [rewind + point["epoch"] for point in points]
        for ax, key in zip(axes, ("test_accuracy", "train_loss", "learning_rate")):
            ax.plot(x, [point[key] for point in points], color=color, label=METHOD_LABELS[method])
        rows.extend({"method": method, "t": rewind + point["epoch"],
                     **{key: point[key] for key in ("test_accuracy", "train_loss", "learning_rate")}}
                    for point in points)
    if not rows:
        plt.close(fig)
        print("No baseline epoch summaries yet; run section 7 first.")
        return None
    for ax, ylabel in zip(axes, ("Test accuracy (%)", "Training loss", "Learning rate")):
        ax.set(xlabel="Completed epoch t", ylabel=ylabel, xlim=(rewind, terminal))
        ax.grid(alpha=0.25)
    axes[0].legend(fontsize=8)
    fig.suptitle(f"Baseline diagnostics — {config['model_type']} / {config['dataset']}, k={rewind}, T={terminal}")
    fig.tight_layout()
    output_dir = Path(output_dir)
    fig.savefig(output_dir / "baseline_learning_curves.png", dpi=160)
    with (output_dir / "baseline_learning_curves.csv").open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=["method", "t", "test_accuracy", "train_loss", "learning_rate"])
        writer.writeheader()
        writer.writerows(rows)
    return fig


In [ ]:
%%writefile act_matching_vgg11.py
from tqdm import tqdm
import numpy as np
import scipy.optimize

import torch
from torch import nn
from torch.cuda.amp import GradScaler, autocast
from torch.nn import CrossEntropyLoss
from torch.optim import SGD, lr_scheduler
import torchvision
import torchvision.transforms as T
from models import VGG11_nofc

def run_corr_matrix(net0, net1, loader, device="cpu"):

    n = len(loader)
    with torch.no_grad():
        net0.eval()
        net1.eval()
        for i, (images, _) in enumerate(tqdm(loader)):
            img_t = images.float().to(device)
            out0 = net0(img_t).double()
            out0 = out0.permute(0, 2, 3, 1).reshape(-1, out0.shape[1])
            out1 = net1(img_t).double()
            out1 = out1.permute(0, 2, 3, 1).reshape(-1, out1.shape[1])

            mean0_b = out0.mean(dim=0)
            mean1_b = out1.mean(dim=0)
            sqmean0_b = out0.square().mean(dim=0)
            sqmean1_b = out1.square().mean(dim=0)
            outer_b = (out0.T @ out1) / out0.shape[0]
            if i == 0:
                mean0 = torch.zeros_like(mean0_b)
                mean1 = torch.zeros_like(mean1_b)
                sqmean0 = torch.zeros_like(sqmean0_b)
                sqmean1 = torch.zeros_like(sqmean1_b)
                outer = torch.zeros_like(outer_b)
            mean0 += mean0_b / n
            mean1 += mean1_b / n
            sqmean0 += sqmean0_b / n
            sqmean1 += sqmean1_b / n
            outer += outer_b / n

    cov = outer - torch.outer(mean0, mean1)
    std0 = (sqmean0 - mean0**2).sqrt()
    std1 = (sqmean1 - mean1**2).sqrt()
    corr = cov / (torch.outer(std0, std1) + 1e-4)
    return corr

def get_layer_perm1(corr_mtx):
    corr_mtx_a = corr_mtx.cpu().numpy()
    row_ind, col_ind = scipy.optimize.linear_sum_assignment(corr_mtx_a, maximize=True)
    assert (row_ind == np.arange(len(corr_mtx_a))).all()
    perm_map = torch.tensor(col_ind).long()
    return perm_map

def get_layer_perm(net0, net1, loader, device):
    corr_mtx = run_corr_matrix(net0, net1, loader, device=device)
    return get_layer_perm1(corr_mtx)

def permute_output(perm_map, layer):
    pre_weights = [layer.weight, layer.bias]

    if hasattr(layer, 'weight_orig'):
        pre_weights.append(layer.weight_orig)
    if hasattr(layer, 'weight_mask'):
        pre_weights.append(layer.weight_mask)

    for w in pre_weights:
        w.data = w[perm_map]

def permute_input(perm_map, layer):
    w = layer.weight
    if isinstance(layer, nn.Linear) and w.shape[1] != len(perm_map):
        if w.shape[1] % len(perm_map) != 0:
            raise ValueError(
                "Linear layer input size must be divisible by the activation "
                "permutation size"
            )
        spatial_size = w.shape[1] // len(perm_map)

        perm_map = (
            perm_map[:, None] * spatial_size
            + torch.arange(spatial_size, device=perm_map.device)[None, :]
        ).reshape(-1)
    perm_map = perm_map.to(w.device)
    w.data = w[:, perm_map]

    if hasattr(layer, 'weight_orig'):
        layer.weight_orig.data = layer.weight_orig[:, perm_map]

    if hasattr(layer, 'weight_mask'):
        layer.weight_mask.data = layer.weight_mask[:, perm_map]

def subnet(model, n_layers):
    return model.features[:n_layers]

def permute_model(model0, model1, modelA_sparse, loader, config):

    device = torch.device(config['device'])
    model0.to(device)
    model1.to(device)
    modelA_sparse.to(device)

    feats1 = modelA_sparse.features
    feats0 = model0.features

    n = len(feats1)
    for i in range(n):
        if not isinstance(feats1[i], nn.Conv2d):
            continue

        assert isinstance(feats1[i + 1], nn.ReLU)
        perm_map = get_layer_perm(subnet(model1, i + 2), subnet(model0, i + 2), loader, device)

        permute_output(perm_map, feats1[i])
        permute_output(perm_map, feats0[i])

        next_layer = None
        next_layer0 = None
        for j in range(i + 1, n):
            if isinstance(feats1[j], nn.Conv2d):
                next_layer = feats1[j]
                next_layer0 = feats0[j]
                break
        if next_layer is None:
            next_layer = modelA_sparse.classifier
            next_layer0 = model0.classifier
        permute_input(perm_map, next_layer)
        permute_input(perm_map, next_layer0)

    return modelA_sparse, model0


In [ ]:
%%writefile act_matching_resnet20.py
from tqdm import tqdm
import numpy as np
import scipy.optimize

import torch
from torch import nn
from torch.cuda.amp import GradScaler, autocast
from torch.nn import CrossEntropyLoss
from torch.optim import SGD, lr_scheduler
import torchvision
import torchvision.transforms as T
from models import ResNet
import yaml

def run_corr_matrix(net0, net1, loader, device="cuda"):

    n = len(loader)
    mean0 = mean1 = std0 = std1 = None
    with torch.no_grad():
        net0.eval()
        net1.eval()
        for i, (images, _) in enumerate(tqdm(loader)):

            img_t = images.float().to(next(net0.parameters()).device)
            out0 = net0(img_t)
            out0 = out0.reshape(out0.shape[0], out0.shape[1], -1).permute(0, 2, 1)
            out0 = out0.reshape(-1, out0.shape[2]).double()

            out1 = net1(img_t)
            out1 = out1.reshape(out1.shape[0], out1.shape[1], -1).permute(0, 2, 1)
            out1 = out1.reshape(-1, out1.shape[2]).double()

            mean0_b = out0.mean(dim=0)
            mean1_b = out1.mean(dim=0)
            std0_b = out0.std(dim=0)
            std1_b = out1.std(dim=0)
            outer_b = (out0.T @ out1) / out0.shape[0]

            if i == 0:
                mean0 = torch.zeros_like(mean0_b)
                mean1 = torch.zeros_like(mean1_b)
                std0 = torch.zeros_like(std0_b)
                std1 = torch.zeros_like(std1_b)
                outer = torch.zeros_like(outer_b)
            mean0 += mean0_b / n
            mean1 += mean1_b / n
            std0 += std0_b / n
            std1 += std1_b / n
            outer += outer_b / n

    cov = outer - torch.outer(mean0, mean1)
    corr = cov / (torch.outer(std0, std1) + 1e-4)
    return corr

def get_layer_perm1(corr_mtx):
    corr_mtx_a = corr_mtx.cpu().numpy()
    row_ind, col_ind = scipy.optimize.linear_sum_assignment(corr_mtx_a, maximize=True)
    assert (row_ind == np.arange(len(corr_mtx_a))).all()
    perm_map = torch.tensor(col_ind).long()
    return perm_map

def get_layer_perm(net0, net1, loader):
    corr_mtx = run_corr_matrix(net0, net1, loader)
    return get_layer_perm1(corr_mtx)

def permute_output(perm_map, conv, bn=None):
    pre_weights = [conv.weight]
    if conv.bias is not None:
        pre_weights.append(conv.bias)
    if bn is not None:
        pre_weights.extend([bn.weight, bn.bias, bn.running_mean, bn.running_var])

    if hasattr(conv, 'weight_orig'):
        pre_weights.append(conv.weight_orig)
    if hasattr(conv, 'weight_mask'):
        pre_weights.append(conv.weight_mask)

    for w in pre_weights:
        w.data = w[perm_map]

def permute_input(perm_map, conv):
    w = conv.weight
    w.data = w[:, perm_map, :, :]

    if hasattr(conv, 'weight_orig'):
        conv.weight_orig.data = conv.weight_orig[:, perm_map, :, :]

    if hasattr(conv, 'weight_mask'):
        conv.weight_mask.data = conv.weight_mask[:, perm_map, :, :]

def get_blocks(net):
    return nn.Sequential(nn.Sequential(net.conv1, net.bn1, nn.ReLU()),
                         *net.layer1, *net.layer2, *net.layer3)

def permute_model_resnet(model0, model1, modelA_sparse, loader, config):

    blocks0 = get_blocks(model0)
    blocks1 = get_blocks(model1)
    blocks_sparse = get_blocks(modelA_sparse)

    for k in range(1, len(blocks0)):
        block0 = blocks0[k]
        block1 = blocks1[k]
        block_sparse = blocks_sparse[k]
        subnet0 = nn.Sequential(blocks0[:k], block0.conv1, block0.bn1, nn.ReLU())
        subnet1 = nn.Sequential(blocks1[:k], block1.conv1, block1.bn1, nn.ReLU())
        perm_map = get_layer_perm(subnet1, subnet0, loader)
        permute_output(perm_map, block0.conv1, block0.bn1)
        permute_input(perm_map, block0.conv2)
        permute_output(perm_map, block_sparse.conv1, block_sparse.bn1)
        permute_input(perm_map, block_sparse.conv2)

    kk = [3, 6, 8]

    perm_map = get_layer_perm(blocks1[:kk[0]+1], blocks0[:kk[0]+1], loader)
    permute_output(perm_map, model0.conv1, model0.bn1)
    permute_output(perm_map, modelA_sparse.conv1, modelA_sparse.bn1)
    for block in model0.layer1:
        permute_input(perm_map, block.conv1)
        permute_output(perm_map, block.conv2, block.bn2)
    for block in modelA_sparse.layer1:
        permute_input(perm_map, block.conv1)
        permute_output(perm_map, block.conv2, block.bn2)
    block = model0.layer2[0]
    permute_input(perm_map, block.conv1)
    permute_input(perm_map, block.shortcut[0])
    block_sparse = modelA_sparse.layer2[0]
    permute_input(perm_map, block_sparse.conv1)
    permute_input(perm_map, block_sparse.shortcut[0])

    perm_map = get_layer_perm(blocks1[:kk[1]+1], blocks0[:kk[1]+1], loader)
    for i, block in enumerate(model0.layer2):
        if i > 0:
            permute_input(perm_map, block.conv1)
        else:
            permute_output(perm_map, block.shortcut[0], block.shortcut[1])
        permute_output(perm_map, block.conv2, block.bn2)
    for i, block in enumerate(modelA_sparse.layer2):
        if i > 0:
            permute_input(perm_map, block.conv1)
        else:
            permute_output(perm_map, block.shortcut[0], block.shortcut[1])
        permute_output(perm_map, block.conv2, block.bn2)
    block = model0.layer3[0]
    permute_input(perm_map, block.conv1)
    permute_input(perm_map, block.shortcut[0])
    block_sparse = modelA_sparse.layer3[0]
    permute_input(perm_map, block_sparse.conv1)
    permute_input(perm_map, block_sparse.shortcut[0])

    perm_map = get_layer_perm(blocks1[:kk[2]+1], blocks0[:kk[2]+1], loader)
    for i, block in enumerate(model0.layer3):
        if i > 0:
            permute_input(perm_map, block.conv1)
        else:
            permute_output(perm_map, block.shortcut[0], block.shortcut[1])
        permute_output(perm_map, block.conv2, block.bn2)
    for i, block in enumerate(modelA_sparse.layer3):
        if i > 0:
            permute_input(perm_map, block.conv1)
        else:
            permute_output(perm_map, block.shortcut[0], block.shortcut[1])
        permute_output(perm_map, block.conv2, block.bn2)
    model0.linear.weight.data = model0.linear.weight[:, perm_map]
    modelA_sparse.linear.weight.data = modelA_sparse.linear.weight[:, perm_map]
    if hasattr(modelA_sparse.linear, 'weight_orig'):

        modelA_sparse.linear.weight_orig.data = modelA_sparse.linear.weight_orig[:, perm_map]

    if hasattr(modelA_sparse.linear, 'weight_mask'):

        modelA_sparse.linear.weight_mask.data = modelA_sparse.linear.weight_mask[:, perm_map]

    return model0, modelA_sparse


In [ ]:
%%writefile resnet50_src/act_matching_resnet50.py
from tqdm import tqdm
import numpy as np
import scipy.optimize

import torch
from torch import nn
import torchvision.transforms as T
from models import get_blocks

def run_corr_matrix(net0, net1, loader, device="cuda"):

    n = len(loader)
    with torch.no_grad():
        net0.eval()
        net1.eval()
        for i, (images, _) in enumerate(tqdm(loader)):

            img_t = images.float().cuda()
            out0 = net0(img_t).double()
            out0 = out0.permute(0, 2, 3, 1).reshape(-1, out0.shape[1])
            out1 = net1(img_t).double()
            out1 = out1.permute(0, 2, 3, 1).reshape(-1, out1.shape[1])

            mean0_b = out0.mean(dim=0)
            mean1_b = out1.mean(dim=0)
            sqmean0_b = out0.square().mean(dim=0)
            sqmean1_b = out1.square().mean(dim=0)
            outer_b = (out0.T @ out1) / out0.shape[0]
            if i == 0:
                mean0 = torch.zeros_like(mean0_b)
                mean1 = torch.zeros_like(mean1_b)
                sqmean0 = torch.zeros_like(sqmean0_b)
                sqmean1 = torch.zeros_like(sqmean1_b)
                outer = torch.zeros_like(outer_b)
            mean0 += mean0_b / n
            mean1 += mean1_b / n
            sqmean0 += sqmean0_b / n
            sqmean1 += sqmean1_b / n
            outer += outer_b / n

    cov = outer - torch.outer(mean0, mean1)
    std0 = (sqmean0 - mean0**2).sqrt()
    std1 = (sqmean1 - mean1**2).sqrt()
    corr = cov / (torch.outer(std0, std1) + 1e-4)
    return corr

def get_layer_perm1(corr_mtx):
    corr_mtx_a = corr_mtx.cpu().numpy()
    corr_mtx_a = np.nan_to_num(corr_mtx_a)
    row_ind, col_ind = scipy.optimize.linear_sum_assignment(corr_mtx_a, maximize=True)
    assert (row_ind == np.arange(len(corr_mtx_a))).all()
    perm_map = torch.tensor(col_ind).long()
    return perm_map

def get_layer_perm(net0, net1, loader):
    corr_mtx = run_corr_matrix(net0, net1, loader)
    return get_layer_perm1(corr_mtx)

def permute_output(perm_map, conv, bn=None):
    pre_weights = [conv.weight]
    if bn is not None:
        pre_weights.extend([bn.weight, bn.bias, bn.running_mean, bn.running_var])

    if hasattr(conv, 'weight_orig'):
        pre_weights.append(conv.weight_orig)
    if hasattr(conv, 'weight_mask'):
        pre_weights.append(conv.weight_mask)

    for w in pre_weights:
        w.data = w[perm_map]

def permute_input(perm_map, layer):
    w = layer.weight
    w.data = w[:, perm_map]

    if hasattr(layer, 'weight_orig'):
        layer.weight_orig.data = layer.weight_orig[:, perm_map]
    if hasattr(layer, 'weight_mask'):
        layer.weight_mask.data = layer.weight_mask[:, perm_map]

def get_permk(k):
    if k == 0:
        return 0
    elif k > 0 and k <= 3:
        return 3
    elif k > 3 and k <= 7:
        return 7
    elif k > 7 and k <= 13:
        return 13
    elif k > 13 and k <= 16:
        return 16
    else:
        raise Exception()

def permute_model_resnet50(model0, model1, modelA_sparse, loader, config):
    last_kk = None
    blocks0 = get_blocks(model0)
    blocks1 = get_blocks(model1)
    blocks_sparse = get_blocks(modelA_sparse)

    for k in range(1, len(blocks0)):
        block0 = blocks0[k]
        block1 = blocks1[k]
        block_sparse = blocks_sparse[k]
        subnet0 = nn.Sequential(blocks0[:k],
                                block0.conv1, block0.bn1, block0.relu)
        subnet1 = nn.Sequential(blocks1[:k],
                                block1.conv1, block1.bn1, block1.relu)

        perm_map = get_layer_perm(subnet1, subnet0, loader)
        permute_output(perm_map, block0.conv1, block0.bn1)
        permute_input(perm_map, block0.conv2)
        permute_output(perm_map, block_sparse.conv1,block_sparse.bn1)
        permute_input(perm_map, block_sparse.conv2)

        subnet0 = nn.Sequential(blocks0[:k],
                                block0.conv1, block0.bn1, block0.relu,
                                block0.conv2, block0.bn2, block0.relu)
        subnet1 = nn.Sequential(blocks1[:k],
                                block1.conv1, block1.bn1, block1.relu,
                                block1.conv2, block1.bn2, block1.relu)
        perm_map = get_layer_perm(subnet1, subnet0, loader)
        permute_output(perm_map, block1.conv2, block1.bn2)
        permute_input(perm_map, block1.conv3)
        permute_output(perm_map, block_sparse.conv2, block_sparse.bn2)
        permute_input(perm_map, block_sparse.conv3)

    for k in range(len(blocks0)):
        kk = get_permk(k)
        if kk != last_kk:
            perm_map = get_layer_perm(blocks1[:kk+1], blocks0[:kk+1], loader)
            last_kk = kk

        if k > 0:
            permute_output(perm_map, blocks0[k].conv3, blocks0[k].bn3)
            shortcut = blocks0[k].downsample
            if shortcut:
                permute_output(perm_map, shortcut[0], shortcut[1])
            permute_output(perm_map, blocks_sparse[k].conv3, blocks_sparse[k].bn3)
            shortcut = blocks_sparse[k].downsample
            if shortcut:
                permute_output(perm_map, shortcut[0], shortcut[1])

        else:
            permute_output(perm_map, model0.conv1, model0.bn1)
            permute_output(perm_map, modelA_sparse.conv1, modelA_sparse.bn1)

        if k+1 < len(blocks0):
            permute_input(perm_map, blocks0[k+1].conv1)
            shortcut = blocks0[k+1].downsample
            if shortcut:
                permute_input(perm_map, shortcut[0])
            permute_input(perm_map, blocks_sparse[k+1].conv1)
            shortcut = blocks_sparse[k+1].downsample
            if shortcut:
                permute_input(perm_map, shortcut[0])
        else:
            permute_input(perm_map, model0.fc)
            permute_input(perm_map, modelA_sparse.fc)

    return model0, modelA_sparse


In [ ]:
%%writefile experiment.py
import argparse
from contextlib import nullcontext
import copy
import datetime
import os
import random
import json

import numpy as np
import yaml
from act_matching_resnet20 import permute_model_resnet
from act_matching_vgg11 import permute_model
from comparison import (
    comparison_settings,
    growth_stage_epochs,
    initialize_dense_model,
    load_rewind_model,
    progressive_budget,
    record_final_result,
    seed_training,
    train_sparse_baselines,
)
from models import *
from progressive import (
    assert_density,
    grow_mask,
    mask_retention_statistics,
    mask_statistics,
    snapshot_masks,
    transfer_mask,
)
from pruning import prune_model
from resnet50_src.act_matching_resnet50 import permute_model_resnet50
from run_logging import finish_run, log_metrics, start_run
from torch.utils.flop_counter import FlopCounterMode
from training import *
from utils import *

def train_model_a_b(seed, config):

    results = {}

    current_time = datetime.datetime.now().strftime("%d %b %Y %H:%M:%S")

    assert hasattr(torch.backends, config["device"]), (
        f"Device {config['device']} not supported"
    )
    device = torch.device(config["device"] if torch.cuda.is_available() else "cpu")

    start_run(
        f"{config['model_type']}_seed_{seed}",
        {
            "project": config["project"],
            "sparsity": config["pruning"]["sparsity"],
            "width": config["width_multiplier"],
            "seed": seed,
        },
        config["save_path"],
    )

    model_A_dense = initialize_dense_model(config, "A") if config.get("comparison", {}).get("enabled", False) else get_model(config)
    trainer(
        model_A_dense,
        config["optimizer"],
        config["epochs"],
        config["batch_size"],
        config["device"],
        "Model_A_Dense",
        sparse=False,
        config=config,
    )
    train_dl, test_dl = cifar_dataloader(config["batch_size"], config)

    results["A_dense"] = evaluate(model_A_dense, test_dl, config["device"])
    print(f"A_dense: {results['A_dense'][0]:.2f}%, {results['A_dense'][1]:.4f}")

    if config["model_type"] == "ResNet50":
        results["A_dense_top5"] = evaluate_top5(
            model_A_dense, test_dl, config["device"]
        )
        print(
            f"A_dense_top5: {results['A_dense_top5'][0]:.2f}%, Loss: {results['A_dense_top5'][1]:.4f}"
        )

    if config["model_type"] in ("ResNet", "VGG11", "ResNet50", "LeNet"):
        model_A_sparse = copy.deepcopy(model_A_dense)
        if config["model_type"] == "ResNet" or config["model_type"] == "ResNet50":
            sparsity_tolerance = config["pruning"]["tolerance_resnet20"]
        else:
            sparsity_tolerance = config["pruning"]["tolerance_vgg11"]
        prune_model(
            config,
            model=model_A_sparse,
            target_sparsity=config["pruning"]["sparsity"],
            optimizer_config=config["optimizer"],
            prune_epochs=config["pruning"]["prune_epochs"],
            initial_lr=config["pruning"]["prune_lr"],
            batch_size=config["batch_size"],
            device=device,
            initial_prune_perc=config["pruning"]["prune_perc"],
            train_epochs_per_prune=config["pruning"]["train_epochs_per_prune"],
            sparsity_tolerance=sparsity_tolerance,
        )
        original_model_A_sparse = copy.deepcopy(model_A_sparse)
        torch.save(
            model_A_sparse,
            config["save_path"]
            + "model_a_sparse_sparsity_"
            + str(config["pruning"]["sparsity"])
            + "_seed_"
            + str(config["seed"]),
        )
        print("\n First hooks check \n")
        check_hooks(original_model_A_sparse)

        for name, module in model_A_sparse.named_modules():
            if isinstance(module, nn.Conv2d):
                if (
                    config["model_type"] == "ResNet"
                    or config["model_type"] == "ResNet50"
                    and "shortcut" in name
                    and "downsample" in name
                ):
                    continue
                assert hasattr(module, "weight_mask")
        results["A_sparse"] = evaluate(model_A_sparse, test_dl, config["device"])
        print(f"A_sparse: {results['A_sparse'][0]:.2f}%, {results['A_sparse'][1]:.4f}")

    model_B_dense = initialize_dense_model(config, "B") if config.get("comparison", {}).get("enabled", False) else get_model(config)
    trainer(
        model_B_dense,
        config["optimizer"],
        config["model_b_epochs"],
        config["batch_size"],
        config["device"],
        "Model_B_Dense",
        sparse=False,
        config=config,
    )
    results["B_dense"] = evaluate(model_B_dense, test_dl, config["device"])
    print(f"B_dense: {results['B_dense'][0]:.2f}%, {results['B_dense'][1]:.4f}")

    if config["model_type"] == "ResNet50":
        results["B_dense_top5"] = evaluate_top5(
            model_A_dense, test_dl, config["device"]
        )
        print(
            f"B_dense_top5: {results['B_dense_top5'][0]:.2f}%, Loss: {results['B_dense_top5'][1]:.4f}"
        )

    validate_permutation = config["model_type"] in ("ResNet", "VGG11", "LeNet")
    if validate_permutation:
        validation_inputs = next(iter(test_dl))[0][:8].to(config["device"])
        model_A_dense.eval()
        model_A_sparse.eval()
        with torch.no_grad():
            dense_before = model_A_dense(validation_inputs).detach().clone()
            sparse_before = model_A_sparse(validation_inputs).detach().clone()
        mask_before = mask_statistics(model_A_sparse, config["model_type"])

    if config["model_type"] == "ResNet":
        permute_model_0, permuted_model_A_sparse = permute_model_resnet(
            model_A_dense, model_B_dense, model_A_sparse, train_dl, config
        )
        permuted_model_a_dense = permute_model_0
        results["A_dense_permuted"] = evaluate(
            permute_model_0, test_dl, config["device"]
        )
        print(
            f"A_dense_permuted: {results['A_dense_permuted'][0]:.2f}%, {results['A_dense_permuted'][1]:.4f}"
        )

        print("\n Pre-reset")
        model_a = evaluate_merged_models(permute_model_0, model_B_dense, 0.5, config)
        train_accuracy, train_loss = evaluate(model_a, train_dl, config["device"])
        test_accuracy, test_loss = evaluate(model_a, test_dl, config["device"])
        print(
            f"(α=0.5): <-- Merged Model\tTrain Accuracy: {train_accuracy}\tTrain Loss: {train_loss:.4f}\t"
            f"Test Accuracy: {test_accuracy}\tTest Loss: {test_loss:.4f}"
        )

        print("\n Post-reset")
        reset_bn_stats(model_a, train_dl)
        results["Merged Model Post reset evaluation"] = evaluate(
            model_a, test_dl, config["device"]
        )
        print(
            f"Merged Model Post reset evaluation: {results['Merged Model Post reset evaluation'][0]:.2f}%, {results['Merged Model Post reset evaluation'][1]:.4f}"
        )

        results["A_sparse_permuted"] = evaluate(
            permuted_model_A_sparse, test_dl, config["device"]
        )
        print(
            f"\n A_sparse_permuted: {results['A_sparse_permuted'][0]:.2f}%, {results['A_sparse_permuted'][1]:.4f}"
        )

        if compare_model_weights(original_model_A_sparse, permuted_model_A_sparse):
            print("\n\n The weights match for all layers.")
        else:
            print("\n\n The weights do not match for all layers.")

    elif config["model_type"] in ("VGG11", "LeNet"):

        permuted_model_A_sparse, permuted_model_a_dense = permute_model(
            model_A_dense, model_B_dense, model_A_sparse, train_dl, config
        )
        for name, module in permuted_model_A_sparse.named_modules():
            if isinstance(module, nn.Conv2d):
                assert hasattr(module, "weight_mask")
        check_hooks(permuted_model_A_sparse)
        results["A_sparse_permuted"] = evaluate(
            permuted_model_A_sparse, test_dl, config["device"]
        )
        print(
            f"A_sparse_permuted: {results['A_sparse_permuted'][0]:.2f}%, {results['A_sparse_permuted'][1]:.4f}"
        )

        results["A_dense_permuted"] = evaluate(
            permuted_model_a_dense, test_dl, config["device"]
        )
        print(
            f"A_dense_permuted: {results['A_dense_permuted'][0]:.2f}%, {results['A_dense_permuted'][1]:.4f}"
        )
        model_a = evaluate_merged_models(
            model_B_dense, permuted_model_a_dense, 0.5, config
        )
        train_accuracy, train_loss = evaluate(model_a, train_dl, config["device"])
        test_accuracy, test_loss = evaluate(model_a, test_dl, config["device"])
        print(
            f"(α=0.5): <-- Merged Model\tTrain Accuracy: {train_accuracy}\tTrain Loss: {train_loss:.4f}\t"
            f"Test Accuracy: {test_accuracy}\tTest Loss: {test_loss:.4f}"
        )

        if compare_model_weights(original_model_A_sparse, permuted_model_A_sparse):
            print("\n\n The weights match for all layers.")
        else:
            print("\n\n The weights do not match for all layers.")

    elif config["model_type"] == "ResNet50":
        num_batches = 50
        batches = []
        for i, (images, labels) in enumerate(train_dl):
            if i >= num_batches:
                break
            batches.append((images, labels))
        batch_size = batches[0][0].shape[0]
        total_data_points = num_batches * batch_size
        print(f"Batch size: {batch_size}")
        print(f"Total data points: {total_data_points}")

        permute_model_0, permuted_model_A_sparse = permute_model_resnet50(
            model_A_dense, model_B_dense, model_A_sparse, batches, config
        )
        torch.save(
            permute_model_0,
            config["save_path"]
            + "permuted_model_0_sparsity_"
            + str(config["pruning"]["sparsity"])
            + "_seed_"
            + str(config["seed"]),
        )
        results["A_dense_permuted"] = evaluate(
            permute_model_0, test_dl, config["device"]
        )
        print(
            f"A_dense_permuted: {results['A_dense_permuted'][0]:.2f}%, {results['A_dense_permuted'][1]:.4f}"
        )

        print("\n Pre-reset")
        model_a = evaluate_merged_models(permute_model_0, model_B_dense, config)
        train_accuracy, train_loss = evaluate(model_a, train_dl, config["device"])
        test_accuracy, test_loss = evaluate(model_a, test_dl, config["device"])
        print(
            f"(α=0.5): <-- Merged Model\tTrain Accuracy: {train_accuracy}\tTrain Loss: {train_loss:.4f}\t"
            f"Test Accuracy: {test_accuracy}\tTest Loss: {test_loss:.4f}"
        )

        print("\n Post-reset")
        reset_bn_stats(model_a, train_dl)
        results["Merged Model Post reset evaluation"] = evaluate(
            model_a, test_dl, config["device"]
        )
        print(
            f"Merged Model Post reset evaluation: {results['Merged Model Post reset evaluation'][0]:.2f}%, {results['Merged Model Post reset evaluation'][1]:.4f}"
        )

        print("\n Another reset (REPAIR)")
        repaired_model_a = repair_merged_model(
            permute_model_0, model_B_dense, model_a, train_dl, config, 0.5
        )
        results["Merged Model with REPAIR"] = evaluate(
            repaired_model_a, test_dl, config["device"]
        )
        print(
            f"Merged Model with REPAIR: {results['Merged Model with REPAIR'][0]:.2f}%, {results['Merged Model with REPAIR'][1]:.4f}"
        )

        results["A_sparse_permuted"] = evaluate(
            permuted_model_A_sparse, test_dl, config["device"]
        )
        print(
            f"\n A_sparse_permuted: {results['A_sparse_permuted'][0]:.2f}%, {results['A_sparse_permuted'][1]:.4f}"
        )

        if compare_model_weights(original_model_A_sparse, permuted_model_A_sparse):
            print("\n\n The weights match for all layers.")
        else:
            print("\n\n The weights do not match for all layers.")

    if validate_permutation:

        permuted_model_a_dense.eval()
        permuted_model_A_sparse.eval()
        with torch.no_grad():
            dense_after = permuted_model_a_dense(validation_inputs)
            sparse_after = permuted_model_A_sparse(validation_inputs)
        torch.testing.assert_close(dense_after, dense_before, rtol=1e-3, atol=1e-3)
        torch.testing.assert_close(sparse_after, sparse_before, rtol=1e-3, atol=1e-3)
        if mask_statistics(permuted_model_A_sparse, config["model_type"]) != mask_before:
            raise AssertionError("Permutation changed the active mask count")
        permutation_check = {
            "dense_max_logit_difference": float((dense_after - dense_before).abs().max().item()),
            "sparse_max_logit_difference": float((sparse_after - sparse_before).abs().max().item()),
            "active_mask_count_preserved": True,
        }
        with open(os.path.join(config["save_path"], "permutation_check.json"), "w", encoding="utf-8") as stream:
            json.dump(permutation_check, stream, indent=2)
        log_metrics({"event": "permutation_check", **permutation_check})

    torch.save(
        permuted_model_A_sparse,
        config["save_path"]
        + "permuted_model_a_sparse_sparsity_"
        + str(config["pruning"]["sparsity"])
        + "_seed_"
        + str(config["seed"]),
    )
    finish_run()

def train_naive_permuted(seed, k, config):

    results = {}

    current_time = datetime.datetime.now().strftime("%d %b %Y %H:%M:%S")

    assert hasattr(torch.backends, config["device"]), (
        f"Device {config['device']} not supported"
    )
    device = torch.device(config["device"] if torch.cuda.is_available() else "cpu")

    start_run(
        f"rewind_{k}_{config['model_type']}_seed_{seed}",
        {
            "project": f"{config['project']}_compare",
            "sparsity": config["pruning"]["sparsity"],
            "rewind_point": k,
            "width": config["width_multiplier"],
            "seed": seed,
        },
        config["save_path"],
    )

    adjusted_sparse_epochs = config["sparse_epochs"] - k
    train_dl, test_dl = cifar_dataloader(config["batch_size"], config)

    path_rewound_point_B_for_naive_and_permuted = f"{config['save_path']}Model_B_Dense_sparsity_{config['pruning']['sparsity']}_seed_{config['seed']}_epoch_{k}"
    with FlopCounterMode(display=False) as mode:
        model_B_dense_init_perm = get_model(config)
        model_B_dense_init_perm.load_state_dict(
            torch.load(path_rewound_point_B_for_naive_and_permuted)
        )

        results["model_B_perm_rewound_point_eval"] = evaluate(
            model_B_dense_init_perm, test_dl, config["device"]
        )
        print(
            f"model_B_perm_rewound_point_eval: {results['model_B_perm_rewound_point_eval'][0]:.2f}%, {results['model_B_perm_rewound_point_eval'][1]:.4f}"
        )

        permuted_model_A_sparse = torch.load(
            config["save_path"]
            + "permuted_model_a_sparse_sparsity_"
            + str(config["pruning"]["sparsity"])
            + "_seed_"
            + str(config["seed"])
        )
        transfer_sparsity_resnet(permuted_model_A_sparse, model_B_dense_init_perm)
        check_hooks(model_B_dense_init_perm)
        print(
            "Sparsity of the init after transfer sparsity: ",
            calculate_overall_sparsity_from_pth(model_B_dense_init_perm),
        )

        trainer(
            model_B_dense_init_perm,
            config["optimizer"],
            adjusted_sparse_epochs,
            config["batch_size"],
            config["device"],
            "Model_B_Permuted",
            config,
            sparse=True,
            training_type="permuted",
        )

        results["B_permuted"] = evaluate(
            model_B_dense_init_perm, test_dl, config["device"]
        )
        print(
            f"B_permuted: {results['B_permuted'][0]:.2f}%, {results['B_permuted'][1]:.4f}"
        )

    print("=" * 50)
    print(f"Total FLOPs: {mode.get_total_flops()}")
    print(f"Global FLOPs: {mode.get_flop_counts()['Global']}")

    finish_run()

def train_progressive(seed, config):

    progressive_config = config.get("progressive", {})
    targets = progressive_config.get("target_densities", [])
    if not targets:
        raise ValueError(
            "progressive.target_densities must contain at least one target"
        )
    if any(not 0.0 <= target <= 1.0 for target in targets):
        raise ValueError("progressive.target_densities must be in [0, 1]")
    if targets != sorted(targets):
        raise ValueError(
            "progressive.target_densities must be monotonically increasing"
        )
    stage_epochs = growth_stage_epochs(config)

    source_path = (
        config["save_path"]
        + "model_a_sparse_sparsity_"
        + str(config["pruning"]["sparsity"])
        + "_seed_"
        + str(seed)
    )
    source_model = torch.load(source_path, map_location=config["device"])
    compare_final = config.get("comparison", {}).get("enabled", False)
    if compare_final:
        model_b = load_rewind_model(config, "B")
        _, total_weights = mask_statistics(source_model, config["model_type"])
        progressive_budget(config, total_weights)
        seed_training(seed)
    else:
        model_b = get_model(config)
    transfer_mask(source_model, model_b, config["model_type"])
    initial_masks = snapshot_masks(model_b, config["model_type"])

    start_run(
        f"progressive_{config['model_type']}_seed_{seed}",
        {
            "project": f"{config['project']}_progressive",
            "method": "progressive",
            "target_densities": targets,
            "seed": seed,
        },
        config["save_path"],
    )
    train_dl, test_dl = cifar_dataloader(config["batch_size"], config)
    generator = torch.Generator()
    generator.manual_seed(int(progressive_config.get("growth_seed", seed)))
    selection = progressive_config.get("selection", "magnitude")
    growth_epochs = 0

    for stage, target_density in enumerate(targets):
        active, total = mask_statistics(model_b, config["model_type"])
        current_density = active / total
        if target_density < current_density:
            raise ValueError(
                f"Stage {stage} target density {target_density} is below "
                f"current density {current_density}"
            )
        if selection == "gradient":
            _compute_progressive_gradients(model_b, train_dl)
        grow_mask(
            model_b,
            config["model_type"],
            target_density,
            selection=selection,
            generator=generator,
        )
        assert_density(model_b, config["model_type"], target_density)
        checkpoint = (
            f"{config['save_path']}progressive_stage_{stage}_density_"
            f"{target_density}_seed_{seed}.pth"
        )
        torch.save(model_b.state_dict(), checkpoint)
        log_metrics(
            {
                "Progressive Stage": stage,
                "Progressive Density": target_density,
                "Progressive Active Weights": int(round(target_density * total)),
            }
        )
        trainer(
            model_b,
            config["optimizer"],
            stage_epochs[stage],
            config["batch_size"],
            config["device"],
            "Progressive",
            config,
            sparse=True,
            training_type="progressive",
        )
        growth_epochs += stage_epochs[stage]
        evaluate_result = evaluate(model_b, test_dl, config["device"])
        print(
            f"Progressive stage {stage}: density={target_density:.6f}, "
            f"accuracy={evaluate_result[0]:.2f}%"
        )

    grown_masks = snapshot_masks(model_b, config["model_type"])
    final_target = config["pruning"]["sparsity"]
    pruning_epochs = prune_model(
        config,
        model=model_b,
        target_sparsity=final_target,
        optimizer_config=config["optimizer"],
        prune_epochs=config["pruning"]["prune_epochs"],
        initial_lr=config["pruning"]["prune_lr"],
        batch_size=config["batch_size"],
        device=torch.device(config["device"] if torch.cuda.is_available() else "cpu"),
        initial_prune_perc=config["pruning"]["prune_perc"],
        train_epochs_per_prune=progressive_config.get(
            "train_epochs_per_prune", config["pruning"]["train_epochs_per_prune"]
        ),
        sparsity_tolerance=config["pruning"].get(
            "tolerance_resnet20"
            if config["model_type"] in ("ResNet", "ResNet50")
            else "tolerance_vgg11",
            0.005,
        ),
    )
    retention = mask_retention_statistics(
        model_b,
        config["model_type"],
        initial_masks,
        grown_masks,
    )
    log_metrics(
        {
            "Progressive Initial Active": retention["initial_active"],
            "Progressive Added Active": retention["added_active"],
            "Progressive Final Active": retention["final_active"],
            "Progressive Initial Retained": retention["initial_retained"],
            "Progressive Added Retained": retention["added_retained"],
            "Progressive Initial Retention": retention["initial_retention"],
            "Progressive Added Retention": retention["added_retention"],
            "Progressive Initial Retention Percent": (
                retention["initial_retention"] * 100
            ),
            "Progressive Added Retention Percent": retention["added_retention"] * 100,
        }
    )
    print(
        "Progressive mask retention: "
        f"initial={retention['initial_retention']:.2%}, "
        f"added={retention['added_retention']:.2%}"
    )
    if compare_final:
        terminal, rewind, minimum_post_prune = comparison_settings(config)
        post_pruning_epochs = terminal - rewind - growth_epochs - pruning_epochs
        if post_pruning_epochs < minimum_post_prune:
            raise ValueError("Insufficient epochs left for post-pruning training; increase comparison.total_epochs")
        optimizer_config = copy.deepcopy(config["optimizer"])
        optimizer_config["sparse_lr"] = config["comparison"].get(
            "post_prune_lr", config["pruning"]["prune_lr"]
        )
        trainer(
            model_b, optimizer_config, post_pruning_epochs, config["batch_size"],
            config["device"], "Progressive_PostPrune", config,
            sparse=True, training_type="progressive",
        )
        record_final_result("progressive", model_b, config, {
            "growth": growth_epochs,
            "pruning": pruning_epochs,
            "post_pruning": post_pruning_epochs,
        }, test_loader=test_dl)
    result = evaluate(model_b, test_dl, config["device"])
    torch.save(
        model_b,
        config["save_path"] + f"model_b_progressive_seed_{seed}.pth",
    )
    print(f"Progressive final accuracy: {result[0]:.2f}%")
    finish_run()

def _compute_progressive_gradients(model, train_dl):

    was_training = model.training
    model.eval()
    model.zero_grad(set_to_none=True)
    inputs, targets = next(iter(train_dl))
    device = next(model.parameters()).device
    loss = torch.nn.CrossEntropyLoss()(model(inputs.to(device)), targets.to(device))
    loss.backward()
    model.train(was_training)

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("-config", type=str)
    parser.add_argument("-seed", type=int)
    parser.add_argument("-pretrain", type=str)
    parser.add_argument("--rewind", type=int)
    parser.add_argument(
        "--method", choices=("baselines", "progressive", "comparison"), default="baselines"
    )
    parser.add_argument("-slurm_tmpdir", type=str)
    args = parser.parse_args()

    with open(args.config, "r") as f:
        config = yaml.safe_load(f)

    config["seed"] = args.seed

    config["save_path"] = (
        f"{config['save_path']}width_{config['width_multiplier']}/sparsity_{config['pruning']['sparsity'] * 100}/"
    )
    if not os.path.exists(config["save_path"]):
        os.makedirs(config["save_path"], exist_ok=True)
    print(config["save_path"])

    print(config)
    torch.manual_seed(args.seed)
    np.random.seed(args.seed)

    print(args.pretrain)
    if args.pretrain == "True":
        train_model_a_b(args.seed, config)
    elif args.pretrain == "False" and args.method == "comparison":
        train_sparse_baselines(args.seed, config)
    elif args.pretrain == "False" and args.method == "baselines":
        train_naive_permuted(args.seed, args.rewind, config)
    elif args.pretrain == "False" and args.method == "progressive":
        with (FlopCounterMode(display=False) if config.get("count_flops", True) else nullcontext()) as mode:
            train_progressive(args.seed, config)
        if mode is not None:
            print("=" * 50)
            print(f"Total FLOPs: {mode.get_total_flops()}")
            print(f"Global FLOPs: {mode.get_flop_counts()['Global']}")
            log_metrics(
                {
                    "Total FLOPs": mode.get_total_flops(),
                    "Global FLOPs": mode.get_flop_counts()["Global"],
                }
            )
    else:
        raise ValueError("incorrect argument")


In [ ]:
%%writefile configs/config_cifar100_resnet20_4_95_overnight.yaml
model_type: ResNet
epochs: 200
model_b_epochs: 200
sparse_epochs: 200
batch_size: 128
num_workers: 2
init_weights: true
project: cifar100_resnet20x4_95_overnight
entity: null
dataset: cifar100
dataset_root:
  cifar100: ./dataset/cifar100
optimizer:
  type: SGD
  sparse_lr: 0.02
  lr: 0.08
  momentum: 0.9
  weight_decay: 0.0005
pruning:
  sparsity: 0.95
  prune_perc: 0.2
  prune_epochs: 20
  prune_lr: 0.01
  train_epochs_per_prune: 50
  tolerance_vgg11: 0.0000005
  tolerance_resnet20: 0.005
progressive:
  target_densities: [0.10, 0.20]
  epochs_per_stage: [20, 20]
  selection: magnitude
  growth_seed: 0
  train_epochs_per_prune: 1
comparison:
  enabled: true
  total_epochs: 200
  rewind_epoch: 25
  min_post_prune_epochs: 1
  post_prune_lr: 0.02
save_legacy_checkpoints: false
count_flops: false
device: cuda
width_multiplier: 4
save_path: ./res/cifar100_resnet20x4_95_overnight/


In [ ]:
%%writefile configs/config_cifar10_vgg_1_80.yaml
model_type: VGG11
epochs: 5
batch_size: 128
model_b_epochs: 1
sparse_epochs: 15
num_workers: 8
init_weights: true
project: cifar10_vgg11
entity: null
dataset: cifar10
dataset_root:
  cifar10: ./dataset/cifar10
  cifar100: ./dataset/cifar100
optimizer:
  type: SGD
  sparse_lr: 0.02
  lr: 0.08
  momentum: 0.9
  weight_decay: 0.0005
pruning:
  sparsity: 0.80
  prune_perc: 0.2
  prune_epochs: 10
  prune_lr: 0.01
  train_epochs_per_prune: 1
  tolerance_vgg11: 0.0000005
  tolerance_resnet20: 0.005
progressive:
  target_densities: [0.3, 0.4]
  epochs_per_stage: 5
  selection: magnitude
  growth_seed: 0
device: cuda
width_multiplier: 1
save_path: ./res/vgg11/


In [ ]:
%%writefile configs/config_cifar10_vgg_1_80_half_budget.yaml
model_type: VGG11
epochs: 100
model_b_epochs: 100
sparse_epochs: 100
batch_size: 128
num_workers: 2
init_weights: true
project: cifar10_vgg11_half_budget
entity: null
dataset: cifar10
dataset_root:
  cifar10: ./dataset/cifar10
optimizer:
  type: SGD
  sparse_lr: 0.02
  lr: 0.08
  momentum: 0.9
  weight_decay: 0.0005
pruning:
  sparsity: 0.80
  prune_perc: 0.2
  prune_epochs: 10
  prune_lr: 0.01
  train_epochs_per_prune: 25
  tolerance_vgg11: 0.0000005
  tolerance_resnet20: 0.005
progressive:
  target_densities: [0.3, 0.4]
  epochs_per_stage: [2, 3]
  selection: magnitude
  growth_seed: 0
  train_epochs_per_prune: 1
comparison:
  enabled: true
  total_epochs: 100
  rewind_epoch: 0
  min_post_prune_epochs: 1
save_legacy_checkpoints: false
device: cuda
width_multiplier: 1
save_path: ./res/vgg11_half_budget/


In [ ]:
%%writefile configs/config_cifar10_vgg_1_80_paper_budget.yaml
model_type: VGG11
epochs: 200
model_b_epochs: 200
sparse_epochs: 200
batch_size: 128
num_workers: 2
init_weights: true
project: cifar10_vgg11_paper_budget
entity: null
dataset: cifar10
dataset_root:
  cifar10: ./dataset/cifar10
optimizer:
  type: SGD
  sparse_lr: 0.02
  lr: 0.08
  momentum: 0.9
  weight_decay: 0.0005
pruning:
  sparsity: 0.80
  prune_perc: 0.2
  prune_epochs: 10
  prune_lr: 0.01
  train_epochs_per_prune: 50
  tolerance_vgg11: 0.0000005
  tolerance_resnet20: 0.005
progressive:
  target_densities: [0.3, 0.4]
  epochs_per_stage: 5
  selection: magnitude
  growth_seed: 0
  train_epochs_per_prune: 1
comparison:
  enabled: true
  total_epochs: 200
  rewind_epoch: 0
  min_post_prune_epochs: 1
save_legacy_checkpoints: false
device: cuda
width_multiplier: 1
save_path: ./res/vgg11_paper_budget/


In [ ]:
%%writefile configs/config_lenet_1_80.yaml
model_type: LeNet
epochs: 5
batch_size: 128
model_b_epochs: 3
sparse_epochs: 30
num_workers: 8
init_weights: true
project: dummy
entity: null
dataset: mnist
dataset_root:
  mnist: ./dataset/mnist
optimizer:
  type: SGD
  sparse_lr: 0.02
  lr: 0.08
  momentum: 0.9
  weight_decay: 0.0005
pruning:
  sparsity: 0.80
  prune_perc: 0.2
  prune_epochs: 10
  prune_lr: 0.01
  train_epochs_per_prune: 1
  tolerance_vgg11: 0.0000005
  tolerance_resnet20: 0.005
progressive:
  target_densities: [0.3]
  epochs_per_stage: 1
  selection: magnitude
  growth_seed: 0
device: cuda
width_multiplier: 1
save_path: ./res/lenet/


In [ ]:
%%writefile configs/config_mnist.yaml
model_type: ResNet
epochs: 1
batch_size: 128
model_b_epochs: 1
sparse_epochs: 1
num_workers: 4
init_weights: False
project: nnn
entity: ucalgary
dataset: mnist
dataset_root:
  mnist: ./dataset/mnist
optimizer:
  type: SGD
  sparse_lr: 0.02
  lr: 0.08
  momentum: 0.9
  weight_decay: 0.0005
pruning:
  sparsity: 0.80
  prune_perc: 0.2
  prune_epochs: 1
  prune_lr: 0.01
  train_epochs_per_prune: 1
  tolerance_vgg11: 0.0000005
  tolerance_resnet20: 0.005
device: cuda
width_multiplier: 1
save_path: ./res/mnist/


In [ ]:
%%writefile configs/config_vgg_1_80.yaml
model_type: VGG11
epochs: 5
batch_size: 128
model_b_epochs: 1
sparse_epochs: 10
num_workers: 8
init_weights: true
project: VGG11
entity: null
dataset: mnist
dataset_root:
  mnist: ./dataset/mnist
optimizer:
  type: SGD
  sparse_lr: 0.02
  lr: 0.08
  momentum: 0.9
  weight_decay: 0.0005
pruning:
  sparsity: 0.80
  prune_perc: 0.2
  prune_epochs: 10
  prune_lr: 0.01
  train_epochs_per_prune: 1
  tolerance_vgg11: 0.0000005
  tolerance_resnet20: 0.005
progressive:
  target_densities: [0.3, 0.4]
  epochs_per_stage: 2
  selection: magnitude
  growth_seed: 0
device: cuda
width_multiplier: 1
save_path: ./res/vgg11/


### Checkpoint-kompatibilitet


In [ ]:
import ast

experiment_path = PROJECT_DIR / "experiment.py"
source = experiment_path.read_text(encoding="utf-8")
lines = source.splitlines(keepends=True)
edits = []
for node in ast.walk(ast.parse(source)):
    if not (isinstance(node, ast.Call) and isinstance(node.func, ast.Attribute)
            and isinstance(node.func.value, ast.Name)
            and node.func.value.id == "torch" and node.func.attr == "load"):
        continue
    if any(keyword.arg == "weights_only" for keyword in node.keywords):
        continue
    state_dict = (isinstance(node.args[0], ast.Name)
                  and node.args[0].id == "path_rewound_point_B_for_naive_and_permuted")

    column = len(lines[node.end_lineno - 1].encode("utf-8")[:node.end_col_offset - 1].decode("utf-8"))
    offset = sum(map(len, lines[:node.end_lineno - 1])) + column
    separator = "" if source[:offset].rstrip().endswith(",") else ","
    edits.append((offset, separator + f" weights_only={state_dict}"))
for offset, insertion in sorted(edits, reverse=True):
    source = source[:offset] + insertion + source[offset:]
compile(source, str(experiment_path), "exec")
experiment_path.write_text(source, encoding="utf-8")
print("Checkpoint-indlæsninger tilpasset:", len(edits))


## 3. Vælg forsøg og lagring


In [ ]:
#@title Forsøgsindstillinger
CONFIG_FILE = "config_cifar100_resnet20_4_95_overnight.yaml" #@param ["config_cifar100_resnet20_4_95_overnight.yaml", "config_cifar10_vgg_1_80.yaml", "config_cifar10_vgg_1_80_half_budget.yaml", "config_cifar10_vgg_1_80_paper_budget.yaml", "config_lenet_1_80.yaml", "config_mnist.yaml", "config_vgg_1_80.yaml"]
SEED = 0 #@param {type:"integer"}
QUICK_RUN = False #@param {type:"boolean"}
RUN_PRETRAIN = True #@param {type:"boolean"}
RUN_PROGRESSIVE = True #@param {type:"boolean"}
RUN_SPARSE_COMPARISON = True #@param {type:"boolean"}
FINAL_T = 0 #@param {type:"integer"}
REWIND = 25 #@param {type:"integer"}
USE_DRIVE = True #@param {type:"boolean"}
RUN_LABEL = "" #@param {type:"string"}
DOWNLOAD_RESULTS = True #@param {type:"boolean"}


In [ ]:
def select_output_root(use_drive, project_dir, *, mountpoint=Path("/content/drive"), drive_module=None):
    local_root = project_dir / "results"
    if use_drive:
        try:
            if drive_module is None:
                from google.colab import drive as drive_module
            drive_module.mount(str(mountpoint))
            my_drive = mountpoint / "MyDrive"
            if not my_drive.is_dir():
                raise OSError("Google Drive blev ikke monteret: MyDrive mangler.")
            drive_root = my_drive / "sparse-rebasin-results"
            drive_root.mkdir(parents=True, exist_ok=True)

            import tempfile
            with tempfile.TemporaryFile(dir=drive_root) as probe:
                probe.write(b"storage check")
                probe.flush()
            print("Lagring: Google Drive (checkpoints og logs gemmes løbende).")
            return drive_root, True
        except Exception as error:
            print("GOOGLE DRIVE ER IKKE TILGÆNGELIG:", type(error).__name__, str(error))
            print("Fortsætter med lokal lagring. Der gemmes IKKE backup på Google Drive.")
            print("Eksisterende Drive-checkpoints er ikke tilgængelige i denne kørsel.")
    local_root.mkdir(parents=True, exist_ok=True)
    print("Lagring: lokal:", local_root)
    print("Hent ZIP-filen i afsnit 10 før runtime slettes; lokale resultater kan ellers gå tabt.")
    return local_root, False


In [ ]:
import datetime
import json
import math
import shutil

config = yaml.safe_load((PROJECT_DIR / "configs" / CONFIG_FILE).read_text())
config["device"] = "cuda"
config["num_workers"] = 2
config["seed"] = SEED
config["dataset_root"][config["dataset"]] = str(PROJECT_DIR / "dataset" / config["dataset"])
config.setdefault("progressive", {
    "target_densities": [0.3], "epochs_per_stage": 1,
    "selection": "magnitude", "growth_seed": SEED,
})
terminal_epoch = FINAL_T if FINAL_T > 0 else config["sparse_epochs"]
config["sparse_epochs"] = terminal_epoch
config["comparison"] = {**config.get("comparison", {}), **{
    "enabled": True, "total_epochs": terminal_epoch, "rewind_epoch": REWIND,
    "min_post_prune_epochs": 1,
}}
config["save_legacy_checkpoints"] = False
if QUICK_RUN:
    config["epochs"] = 1
    config["model_b_epochs"] = 1
    REWIND = 0
    config["comparison"]["rewind_epoch"] = 0
    print("QUICK_RUN bruger k=0 (den normale overnight-profil bruger k=25).")
    config["progressive"]["epochs_per_stage"] = 1
    config["pruning"]["train_epochs_per_prune"] = 1
    config["progressive"]["train_epochs_per_prune"] = 1

target_sparsity = config["pruning"]["sparsity"]
prune_fraction = config["pruning"]["prune_perc"]
if not (0 < target_sparsity < 1 and 0 < prune_fraction < 1):
    raise ValueError("Denne notebook kræver sparsity og prune_perc mellem 0 og 1.")

if RUN_PROGRESSIVE or RUN_SPARSE_COMPARISON:
    minimum_rounds = math.ceil(math.log1p(-target_sparsity) / math.log1p(-prune_fraction)) + 1
    if config["pruning"]["prune_epochs"] < minimum_rounds:
        print("prune_epochs hæves til", minimum_rounds, "for at nå målsparsiteten.")
        config["pruning"]["prune_epochs"] = minimum_rounds
if RUN_PROGRESSIVE:
    targets = config["progressive"]["target_densities"]
    if not targets or targets != sorted(targets) or any(not 1 - target_sparsity < d <= 1 for d in targets):
        raise ValueError("Growth densities skal være stigende, højere end startdensiteten og højst 1.")
if not 0 <= REWIND <= min(config["epochs"], config["model_b_epochs"]):
    raise ValueError("REWIND skal være mellem 0 og antal dense epoker for både A og B.")
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
from comparison import comparison_settings, progressive_budget, pruning_rounds_required
from progressive import eligible_modules
from utils import get_model

budget_model = get_model({**config, "device": "cpu"})
total_weights = sum(module.weight.numel() for _, module in eligible_modules(budget_model, config["model_type"]))
del budget_model
if QUICK_RUN:
    rounds = pruning_rounds_required(
        round(config["progressive"]["target_densities"][-1] * total_weights), total_weights,
        target_sparsity, prune_fraction,
    ) if RUN_PROGRESSIVE else 0
    quick_terminal = REWIND + max(1, len(config["progressive"]["target_densities"]) + rounds + 1)
    config["comparison"]["total_epochs"] = quick_terminal
    config["sparse_epochs"] = quick_terminal
    print("QUICK_RUN bruger fælles T =", quick_terminal)
comparison_settings(config)
if RUN_PROGRESSIVE:
    growth_epochs, progressive_prune_epochs = progressive_budget(config, total_weights)

source_rounds = pruning_rounds_required(total_weights, total_weights, target_sparsity, prune_fraction)
actual_terminal, actual_rewind, _ = comparison_settings(config)
training_plan = {
    "dense_A_epochs": config["epochs"], "dense_B_epochs": config["model_b_epochs"],
    "source_A_pruning_rounds": source_rounds,
    "source_A_pruning_training_epochs": source_rounds * config["pruning"]["train_epochs_per_prune"],
    "sparse_terminal_epoch": actual_terminal, "completed_dense_epochs_at_rewind": actual_rewind,
    "sparse_epochs_per_baseline": actual_terminal - actual_rewind,
}
if RUN_PROGRESSIVE:
    training_plan["progressive_stage_epochs"] = {
        "growth": growth_epochs, "pruning": progressive_prune_epochs,
        "post_pruning": actual_terminal - actual_rewind - growth_epochs - progressive_prune_epochs,
    }
training_plan["planned_training_epochs_this_run"] = (
    (config["epochs"] + config["model_b_epochs"]
     + training_plan["source_A_pruning_training_epochs"] if RUN_PRETRAIN else 0)
    + (actual_terminal - actual_rewind if RUN_PROGRESSIVE else 0)
    + (3 * (actual_terminal - actual_rewind) if RUN_SPARSE_COMPARISON else 0)
)
print("Planlagt træning:", json.dumps(training_plan, indent=2))

output_root, DRIVE_ACTIVE = select_output_root(USE_DRIVE, PROJECT_DIR)
label = RUN_LABEL.strip() or (
    f"{config['model_type']}_{config['dataset']}_seed{SEED}_"
    + datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d_%H%M%S_UTC")
)
if label in {".", ".."} or "/" in label or chr(92) in label:
    raise ValueError("RUN_LABEL skal være et mappenavn uden skråstreger.")
BASE_RUN_DIR = output_root / label
BASE_RUN_DIR.mkdir(parents=True, exist_ok=True)
config["save_path"] = str(BASE_RUN_DIR) + "/"
CONFIG_PATH = BASE_RUN_DIR / "colab_config.yaml"

if CONFIG_PATH.exists() and yaml.safe_load(CONFIG_PATH.read_text()) != config:
    raise ValueError("Resultatmappen har en anden konfiguration. Vælg et nyt RUN_LABEL.")
CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
(BASE_RUN_DIR / "training_plan.json").write_text(json.dumps(training_plan, indent=2), encoding="utf-8")
RUN_DIR = BASE_RUN_DIR / f"width_{config['width_multiplier']}" / f"sparsity_{target_sparsity * 100}"
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(yaml.safe_dump(config, sort_keys=False))
print("RUN_LABEL:", label)
print("Resultater:", RUN_DIR)


## 4. Kontrollér imports og gem forsøgets kildekode


In [ ]:
subprocess.run([sys.executable, "experiment.py", "--help"], cwd=PROJECT_DIR, check=True)
SOURCE_FILES = ['models.py', 'loaders.py', 'run_logging.py', 'utils.py', 'training.py', 'pruning.py', 'progressive.py', 'comparison.py', 'comparison_plot.py', 'act_matching_vgg11.py', 'act_matching_resnet20.py', 'resnet50_src/act_matching_resnet50.py', 'experiment.py', 'configs/config_cifar100_resnet20_4_95_overnight.yaml', 'configs/config_cifar10_vgg_1_80.yaml', 'configs/config_cifar10_vgg_1_80_half_budget.yaml', 'configs/config_cifar10_vgg_1_80_paper_budget.yaml', 'configs/config_lenet_1_80.yaml', 'configs/config_mnist.yaml', 'configs/config_vgg_1_80.yaml']
for name in SOURCE_FILES:
    target = BASE_RUN_DIR / "source" / name
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(PROJECT_DIR / name, target)
environment = {
    "python": sys.version, "torch": torch.__version__,
    "torchvision": torchvision.__version__, "numpy": np.__version__,
    "scipy": scipy.__version__, "pyyaml": yaml.__version__,
    "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0),
    "drive_requested": USE_DRIVE, "drive_active": DRIVE_ACTIVE,
    "output_root": str(output_root),
}
(BASE_RUN_DIR / "environment.json").write_text(json.dumps(environment, indent=2), encoding="utf-8")

def require_checkpoints(*names):
    missing = [str(RUN_DIR / name) for name in names if not (RUN_DIR / name).is_file()]
    if missing:
        raise FileNotFoundError("Kør fortræningen først. Mangler: " + ", ".join(missing))

def run_experiment(stage, *arguments):
    command = [sys.executable, "-u", "experiment.py", "-config", str(CONFIG_PATH),
               "-seed", str(SEED), *arguments]
    log_path = BASE_RUN_DIR / f"{stage}.log"
    print("Kører:", " ".join(command), flush=True)
    with log_path.open("a", encoding="utf-8") as log:
        log.write("\nCOMMAND: " + " ".join(command) + "\n")
        with subprocess.Popen(command, cwd=PROJECT_DIR, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                              errors="replace", bufsize=1) as process:
            try:
                for line in process.stdout:
                    print(line, end="", flush=True)
                    log.write(line)
                    log.flush()
                returncode = process.wait()
            except BaseException:
                process.terminate()
                process.wait()
                raise
    if returncode:
        raise RuntimeError(f"{stage} fejlede (exit {returncode}). Log: {log_path}")
    print("Afsluttet:", stage)


## 5. Fortræning: dense A -> pruning -> dense B -> permutation


In [ ]:
if not all(name in globals() for name in ("RUN_PRETRAIN", "run_experiment", "config", "RUN_DIR")):
    raise RuntimeError("Opsætningen mangler. Kør først afsnit 1–4 i rækkefølge, især efter en runtime-genstart.")
if RUN_PRETRAIN:
    run_experiment("pretrain", "-pretrain", "True")
else:
    print("Fortræning sprunget over; efterfølgende trin kræver eksisterende checkpoints.")


## 6. Progressive mask growth


In [ ]:
if not all(name in globals() for name in ("RUN_PROGRESSIVE", "run_experiment", "require_checkpoints", "config", "RUN_DIR")):
    raise RuntimeError("Opsætningen mangler. Kør først afsnit 1–4 i rækkefølge, især efter en runtime-genstart.")
if RUN_PROGRESSIVE:
    require_checkpoints(
        f"model_a_sparse_sparsity_{target_sparsity}_seed_{SEED}",
        f"Model_B_Dense_rewind_{REWIND}_seed_{SEED}.pth",
    )
    run_experiment("progressive", "-pretrain", "False", "--method", "progressive")
else:
    print("Progressive growth er slået fra.")


## 7. Figur 2: Naive, Permuted og LTH ved t=T


In [ ]:
if not all(name in globals() for name in ("RUN_SPARSE_COMPARISON", "run_experiment", "require_checkpoints", "config", "RUN_DIR")):
    raise RuntimeError("Opsætningen mangler. Kør først afsnit 1–4 i rækkefølge, især efter en runtime-genstart.")
if RUN_SPARSE_COMPARISON:
    require_checkpoints(
        f"model_a_sparse_sparsity_{target_sparsity}_seed_{SEED}",
        f"permuted_model_a_sparse_sparsity_{target_sparsity}_seed_{SEED}",
        f"Model_A_Dense_rewind_{REWIND}_seed_{SEED}.pth",
        f"Model_B_Dense_rewind_{REWIND}_seed_{SEED}.pth",
    )
    run_experiment("sparse_comparison", "-pretrain", "False", "--method", "comparison")
else:
    print("De tre sparse baselines er slået fra.")


## 8. Se målinger og træningskurver


In [ ]:
from collections import defaultdict
import matplotlib.pyplot as plt

series = defaultdict(list)
for path in sorted(RUN_DIR.glob("*_metrics.jsonl")):
    if path.name not in {
        f"{config['model_type']}_seed_{SEED}_metrics.jsonl",
        f"progressive_{config['model_type']}_seed_{SEED}_metrics.jsonl",
    }:
        continue
    print("\n", path.name)
    records = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
    for record in records:
        for key, value in record.items():
            if "Test Accuracy" in key and isinstance(value, (int, float)):
                label_key = key.replace("Model A Sparse", "Progressive pruning") if path.stem.startswith("progressive_") else key
                series[(path.stem, label_key)].append(value)
    retention = [record for record in records if "Progressive Initial Retention Percent" in record]
    if retention:
        print("Bevarede oprindelige forbindelser (%):", retention[-1]["Progressive Initial Retention Percent"])
        print("Bevarede tilføjede forbindelser (%):", retention[-1]["Progressive Added Retention Percent"])
if series:
    fig, ax = plt.subplots(figsize=(11, 5))
    for (run, metric), values in series.items():
        ax.plot(range(1, len(values) + 1), values, marker="o", markersize=3, label=f"{run}: {metric}")
        print(f"{metric}: seneste test accuracy = {values[-1]:.2f}%")
    ax.set(xlabel="Logget trin", ylabel="Test accuracy (%)", title="Sparse Rebasin")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8, loc="best")
    fig.tight_layout()
    fig.savefig(BASE_RUN_DIR / "accuracy.png", dpi=150)
    plt.show()
else:
    print("Ingen træningsmålinger fundet endnu.")


### Ekstra diagnostik til afsnit 8: lærer baselines stadig?


In [ ]:
from comparison_plot import plot_baseline_learning_curves

baseline_figure = plot_baseline_learning_curves(RUN_DIR, config, BASE_RUN_DIR)
if baseline_figure is not None:
    plt.show()


## 9. Ekstra graf: de fire færdigtrænede modeller ved t=T


In [ ]:
from comparison_plot import plot_final_comparison

if RUN_PROGRESSIVE and RUN_SPARSE_COMPARISON:
    comparison_figure, final_results = plot_final_comparison(RUN_DIR, config, BASE_RUN_DIR)
    plt.show()
else:
    print("Slå RUN_PROGRESSIVE og RUN_SPARSE_COMPARISON til, og kør afsnit 6 og 7 for alle fire slutresultater.")


## 10. Pak resultater til download


In [ ]:
import datetime
import zipfile

def reserve_export_file(directory, stem):
    directory.mkdir(parents=True, exist_ok=True)
    counter = 0
    while True:
        suffix = f"_{counter}" if counter else ""
        candidate = directory / f"{stem}{suffix}.zip"
        try:
            return candidate, candidate.open("xb")
        except FileExistsError:
            counter += 1

export_stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d_%H%M%S_%f_UTC")
archive_path, archive_stream = reserve_export_file(Path("/content"), f"{label}_results_{export_stamp}")
with archive_stream:
    with zipfile.ZipFile(archive_stream, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for result_file in sorted(BASE_RUN_DIR.rglob("*")):
            if result_file.is_file():
                archive.write(result_file, result_file.relative_to(BASE_RUN_DIR))
if DRIVE_ACTIVE:
    drive_archive_path, drive_stream = reserve_export_file(BASE_RUN_DIR.parent, archive_path.stem)
    with drive_stream, archive_path.open("rb") as source_stream:
        shutil.copyfileobj(source_stream, drive_stream)
    print("ZIP på Drive:", drive_archive_path)
print("ZIP:", archive_path, f"({archive_path.stat().st_size / 2**20:.1f} MiB)")
if DOWNLOAD_RESULTS:
    from google.colab import files
    files.download(str(archive_path))
